In [0]:
CREATE OR REPLACE TEMPORARY VIEW global_runtime_dates AS
SELECT 
    /* ELAPRASE */
    DATE('2020-08-01') AS elaprase_dx_start_date,
    DATE('2023-08-01') AS elaprase_tx_start_date,

    /* AVLAYAH */
    -- DATE('2022-08-01') AS avlayah_dx_start_date,
    DATE('2026-03-01') AS avlayah_tx_start_date;

In [0]:
CREATE OR REPLACE TEMP VIEW runtime_parameters AS

SELECT
    (SELECT MAX(service_date) FROM com_edp_prd.com_raw.kom_medical_events) AS max_medical_date,

    (SELECT MAX(fill_date) FROM com_edp_prd.com_raw.kom_pharmacy_events) AS max_pharmacy_date,

    LAST_DAY(
        ADD_MONTHS(
            LEAST(
                (SELECT MAX(service_date) FROM com_edp_prd.com_raw.kom_medical_events),
                (SELECT MAX(fill_date) FROM com_edp_prd.com_raw.kom_pharmacy_events)
            ), -1
        )
    ) AS end_date,

    CURRENT_DATE() AS run_date;

    SELECT * FROM runtime_parameters;

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.payer_rollup_dim AS

WITH base AS (

SELECT DISTINCT
    CAST(payer_id AS STRING) AS payer_id,
    PAYER_NAME,

CASE

WHEN UPPER(PAYER_NAME) RLIKE 'UNITED|OPTUM'
    THEN 'UHC/Optum'

WHEN UPPER(PAYER_NAME) RLIKE 'AETNA|CVS'
    THEN 'Aetna/CVS'

WHEN UPPER(PAYER_NAME) RLIKE 'CIGNA|ESI|EVERNORTH'
    THEN 'Cigna/ESI'

WHEN UPPER(PAYER_NAME) RLIKE 'ANTHEM|ELEVANCE|CARELON'
    THEN 'Elevance/Carelon'

WHEN UPPER(PAYER_NAME) RLIKE 'ILLINOIS|TEXAS|OKLAHOMA|NEW MEXICO'
    THEN 'Prime Therapeutics / HCSC'

ELSE PAYER_NAME

END AS payer_display_name

FROM com_edp_prd.com_raw.kom_plans
),

rollup_ids AS (

SELECT
    payer_display_name,
    DENSE_RANK() OVER (ORDER BY payer_display_name) + 1000 AS payer_display_id
FROM (
    SELECT DISTINCT payer_display_name
    FROM base
)
)

SELECT
    b.PAYER_ID AS source_payer_id,
    b.PAYER_NAME AS source_payer_name,
    r.payer_display_id,
    r.payer_display_name,
    r.payer_display_id AS canonical_payer_id,
    r.payer_display_name AS canonical_payer_name
FROM base b
LEFT JOIN rollup_ids r
ON b.payer_display_name = r.payer_display_name;

### Total Lives

In [0]:
SELECT COUNT(*)
FROM com_edp_prd.com_raw.kom_plans p
LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.payer_rollup_dim r
ON p.PAYER_ID = r.source_payer_id
WHERE r.source_payer_id IS NULL

In [0]:
/* 
PURPOSE
- Create a temporary view (total_lives) that calculates distinct patient lives 
  across Territory and Payer with subtotal and national rollups.

BUSINESS LOGIC
- Combine medical and pharmacy claims into one unified claim universe.
- Attribute each claim to an NPI and Plan.
- Map NPI → Provider ZIP → Territory.
- Map Plan → Payer.
- Count DISTINCT patients.
- Produce 4 rollup levels:
    1) Territory + Payer
    2) Payer (All Territories)
    3) Territory (All Payers)
    4) National Total
*/

CREATE OR REPLACE TEMPORARY VIEW total_lives AS

WITH all_claims AS (
  /* 
    Combine medical and pharmacy events into a single claims dataset.
    Each row represents a distinct patient + NPI + plan + claim combination.
  */
  SELECT DISTINCT * FROM (
    
    /* Medical claims: use Rendering NPI, else Referring NPI */
    SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
      KH_PLAN_ID AS plan_id,
      MEDICAL_EVENT_ID AS claim_id
    FROM com_edp_prd.com_raw.kom_medical_events

    UNION ALL

    /* Pharmacy claims: use Prescriber NPI and primary/secondary plan */
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS NPI,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS plan_id,
      PHARMACY_EVENT_ID AS claim_id
    FROM com_edp_prd.com_raw.kom_pharmacy_events

    -- UNION

    -- /* Medical claims using Rendering NPI */
    -- SELECT DISTINCT 
    --   PATIENT_ID,
    --   RENDERING_NPI AS NPI,
    --   KH_PLAN_ID AS plan_id,
    --   MEDICAL_EVENT_ID AS claim_id
    -- FROM com_edp_prd.com_raw.kom_medical_events
  )
),

-- tagging_zip AS (
--   /* Attach provider ZIP based on NPI */
--   SELECT 
--     a.*, 
--     b.PROVIDER_ZIP AS hcp_zip
--   FROM all_claims a
--   LEFT JOIN com_raw.kom_providers b
--     ON a.npi = b.NPI 
--    AND b.PROVIDER_TYPE = 'INDIVIDUAL'
-- ),

-- tagging_territory AS (
--   /* Map provider ZIP to territory */
--   SELECT 
--     a.*, 
--     COALESCE(CAST(b.territory_id AS STRING), 'Unknown') AS territory_id,
--     COALESCE(b.territory_name, 'Unknown') AS territory
--   FROM tagging_zip a
--   LEFT JOIN cmpa_insights_internal_schema.zip_to_territory_mapping b
--     ON TRY_CAST(a.hcp_zip AS STRING) = TRY_CAST(b.zipcode AS STRING)
-- ),

tagging_zip_v1 AS (
  SELECT 
    a.*,
    LPAD(COALESCE(
      NULLIF(b.hco_zip, '-'), 
      NULLIF(b.hcp_zip, '-'), 
      c.PROVIDER_ZIP
    ), 5, '0') AS final_zip
  FROM all_claims AS a
  LEFT JOIN cmpa_insights_internal_schema.reference_file AS b ON a.npi = b.hcp_npi
  LEFT JOIN com_raw.kom_providers AS c ON a.npi = c.npi AND c.PROVIDER_TYPE = 'INDIVIDUAL'
),

tagging_territory AS (
  /* Map provider ZIP to territory */
  SELECT 
    a.*, 
    COALESCE(CAST(b.territory_id AS STRING), 'Unknown') AS territory_id,
    COALESCE(b.territory_name, 'Unknown') AS territory
  FROM tagging_zip_v1 a
  LEFT JOIN cmpa_insights_internal_schema.zip_to_territory_mapping b
    ON a.final_zip = LPAD(TRY_CAST(b.zipcode AS STRING), 5, '0')
),

tagging_payer AS (
  SELECT 
    a.*,
    COALESCE(CAST(r.canonical_payer_id AS STRING), CAST(b.PAYER_ID AS STRING)) AS PAYER_ID,
    COALESCE(r.canonical_payer_name, b.PAYER_NAME) AS PAYER_NAME
  FROM tagging_territory a
  LEFT JOIN com_raw.kom_plans b
      ON a.plan_id = b.KH_PLAN_ID
  LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.payer_rollup_dim r
      ON b.PAYER_ID = r.source_payer_id
),

agg AS (
  /* 
    Aggregate distinct patients and create rollup levels 
    using GROUPING SETS.
  */
  SELECT
    COALESCE(territory_id, 'ALL Territories') AS territory_id,
    COALESCE(territory, 'All Territories') AS territory,
    COALESCE(payer_id, 'ALL Payers') AS payer_id,
    COALESCE(payer_name, 'All Payers') AS payer_name,
    COUNT(DISTINCT patient_id) AS total_lives,
    CASE
      WHEN GROUPING(territory_id) = 0 AND GROUPING(payer_id) = 0 THEN 'TERRITORY_PAYER'
      WHEN GROUPING(territory_id) = 1 AND GROUPING(payer_id) = 0 THEN 'PAYER_ALL_TERRITORY'
      WHEN GROUPING(territory_id) = 0 AND GROUPING(payer_id) = 1 THEN 'TERRITORY_ALL_PAYER'
      WHEN GROUPING(territory_id) = 1 AND GROUPING(payer_id) = 1 THEN 'NATIONAL'
    END AS rollup_level
  FROM tagging_payer
  GROUP BY GROUPING SETS (
    (territory_id, territory, payer_id, payer_name),
    (payer_id, payer_name),
    (territory_id, territory),
    ()
  )
)

/* Final output ordered by rollup level and total lives */
SELECT *
FROM agg
ORDER BY rollup_level, total_lives DESC;

In [0]:
select * from total_lives;

In [0]:
-- National patients should be stable (or only change if your previous logic was inflating/deflating)
SELECT total_lives
FROM total_lives
WHERE rollup_level = 'NATIONAL';

In [0]:
-- Unknown territory volume should DROP relative to the 3rd-UNION version (because you're avoiding spurious Unknown attributions)
SELECT total_lives
FROM total_lives
WHERE rollup_level = 'TERRITORY_ALL_PAYER'
  AND territory_id = 'Unknown';

In [0]:
-- /* Self-contained proof: builds eligible cohort + constructs NPI by claim type */

-- WITH dx_claims AS (
--   /* Specified Dx claims (E761) from medical + pharmacy */
--   SELECT
--     patient_id,
--     service_date AS fill_date
--   FROM com_edp_prd.com_raw.kom_medical_events
--   WHERE diagnosis_codes LIKE '%E761%'
--     AND service_date BETWEEN '2020-04-01' AND '2025-03-31'

--   UNION ALL

--   SELECT
--     patient_id,
--     fill_date
--   FROM com_edp_prd.com_raw.kom_pharmacy_events
--   WHERE diagnosis_code = 'E761'
--     AND transaction_status = 'PAID'
--     AND fill_date BETWEEN '2020-04-01' AND '2025-03-31'
-- ),

-- eligible_patients AS (
--   /* Eligible = >=2 distinct claim dates with E761 in window */
--   SELECT patient_id
--   FROM dx_claims
--   GROUP BY patient_id
--   HAVING COUNT(DISTINCT fill_date) >= 2
-- ),

-- rebuilt_claims AS (
--   /* MEDICAL rows: NPI = rendering/referring */
--   SELECT DISTINCT
--     me.patient_id,
--     me.medical_event_id AS event_id,
--     me.service_date     AS fill_date,
--     'MEDICAL_EVENTS'    AS source_type,
--     COALESCE(me.rendering_npi, me.referring_npi) AS npi,
--     me.rendering_npi,
--     me.referring_npi,
--     NULL::STRING AS prescriber_npi
--   FROM com_edp_prd.com_raw.kom_medical_events me
--   JOIN eligible_patients ep
--     ON me.patient_id = ep.patient_id
--   WHERE me.service_date BETWEEN '2020-04-01' AND '2025-03-31'

--   UNION ALL

--   /* PHARMACY rows: NPI = prescriber */
--   SELECT DISTINCT
--     pe.patient_id,
--     pe.pharmacy_event_id AS event_id,
--     pe.fill_date         AS fill_date,
--     'PHARMACY_EVENTS'    AS source_type,
--     pe.prescriber_npi    AS npi,
--     NULL::STRING AS rendering_npi,
--     NULL::STRING AS referring_npi,
--     pe.prescriber_npi
--   FROM com_edp_prd.com_raw.kom_pharmacy_events pe
--   JOIN eligible_patients ep
--     ON pe.patient_id = ep.patient_id
--   WHERE pe.fill_date BETWEEN '2020-04-01' AND '2025-03-31'
--     AND pe.transaction_result = 'PAID'
-- )

-- SELECT
--   source_type,
--   COUNT(*) AS rows,
--   COUNT(DISTINCT patient_id) AS patients,
--   COUNT_IF(npi IS NULL) AS null_npi_rows,
--   /* Proof checks */
--   COUNT_IF(source_type='PHARMACY_EVENTS' AND npi = prescriber_npi) AS pharmacy_npi_is_prescriber,
--   COUNT_IF(source_type='MEDICAL_EVENTS'  AND npi = COALESCE(rendering_npi, referring_npi)) AS medical_npi_is_render_or_ref
-- FROM rebuilt_claims
-- GROUP BY 1
-- ORDER BY rows DESC;

### Base Tables

In [0]:
/*
PURPOSE
- Identify eligible patients based on diagnosis and treatment criteria.
- Pull all diagnosis and treatment claims for those eligible patients.

  BUSINESS LOGIC SUMMARY
  1) Pull diagnosis claims (E761, E763).
  2) Pull treatment claims (specific NDCs and procedure codes).
  3) Identify:
    - E761 patients with ≥2 diagnosis dates + at least one treatment → specified_patients.
    - E763 patients with ≥2 diagnosis dates + Elaprase treatment,
      excluding already specified patients → incremental_patients.
  4) Eligible patients = specified + incremental.
  5) Return all diagnosis and treatment claims for eligible patients.
  */


/* ============================================================
   1) ALL DIAGNOSIS CLAIMS (E761 / E763)
   ============================================================ */
CREATE OR REPLACE TEMPORARY VIEW all_dx_claims AS
/* Medical diagnosis claims */
SELECT DISTINCT
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,  -- Provider attribution
    SERVICE_DATE AS FILL_DATE,
    MEDICAL_EVENT_ID AS claim_id,
    KH_PLAN_ID AS plan_id,
    'MEDICAL' AS CLAIM_SOURCE,
    'MEDICAL' AS TRANSACTION_STATUS
FROM com_edp_prd.com_raw.kom_medical_events
WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)

UNION

/* Pharmacy diagnosis claims (paid only) */
SELECT DISTINCT
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    PHARMACY_EVENT_ID AS claim_id,
    COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS plan_id,
    'PHARMACY' AS CLAIM_SOURCE,
    TRANSACTION_RESULT AS TRANSACTION_STATUS
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE DIAGNOSIS_CODE IN ('E761', 'E763')
  AND TRANSACTION_STATUS = 'PAID'
  AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters);



/* ============================================================
   2) ALL TREATMENT CLAIMS
   ============================================================ */
CREATE OR REPLACE TEMPORARY VIEW all_tx_claims AS

/* Medical NDC-based treatment */
SELECT DISTINCT
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    SERVICE_DATE AS FILL_DATE,
    MEDICAL_EVENT_ID AS claim_id,
    NDC11 AS CODE,
    KH_PLAN_ID AS plan_id,
    'MEDICAL' AS CLAIM_SOURCE,
    'MEDICAL' AS TRANSACTION_STATUS
FROM com_edp_prd.com_raw.kom_medical_events
WHERE NDC11 IN ('54092070001', '540920700','8497600101')
  AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)

UNION

/* Medical procedure-based treatment */
SELECT DISTINCT
    PATIENT_ID,
    RENDERING_NPI AS NPI,
    SERVICE_DATE AS FILL_DATE,
    MEDICAL_EVENT_ID AS claim_id,
    PROCEDURE_CODE AS CODE,
    KH_PLAN_ID AS plan_id,
    'MEDICAL' AS CLAIM_SOURCE,
    'MEDICAL' AS TRANSACTION_STATUS
FROM com_edp_prd.com_raw.kom_medical_events
WHERE PROCEDURE_CODE IN ('99601', '99602', '96365', '96366', 'J1743',
                         'S9357', 'S9379', '38206', '38230', '38232',
                         '38240', '38241', '38242', '38243', '38250')
  AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)

UNION

/* Pharmacy treatment (paid only) */
SELECT DISTINCT
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    PHARMACY_EVENT_ID AS claim_id,
    NDC11 AS CODE,
    COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS plan_id,
    'PHARMACY' AS CLAIM_SOURCE,
    TRANSACTION_RESULT AS TRANSACTION_STATUS
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE NDC11 IN ('54092070001', '540920700','8497600101')
  AND TRANSACTION_RESULT = 'PAID'
  AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)

UNION

/* Medical procedure-based treatment */
SELECT DISTINCT
    PATIENT_ID,
    RENDERING_NPI AS NPI,
    SERVICE_DATE AS FILL_DATE,
    MEDICAL_EVENT_ID AS claim_id,
    PROCEDURE_CODE AS CODE,
    KH_PLAN_ID AS plan_id,
    'MEDICAL' AS CLAIM_SOURCE,
    'MEDICAL' AS TRANSACTION_STATUS
FROM com_edp_prd.com_raw.kom_medical_events
WHERE PROCEDURE_CODE IN ('J3490','J3590','J9999')
  AND SERVICE_DATE BETWEEN '2026-03-01' AND (SELECT end_date FROM runtime_parameters);



/* ============================================================
   3) E761 PATIENTS WITH ≥2 DIAGNOSIS DATES
   ============================================================ */
CREATE OR REPLACE TEMPORARY VIEW e761_patients_2dx AS
SELECT PATIENT_ID
FROM (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)

    UNION

    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;



/* ============================================================
   4) SPECIFIED PATIENTS
   - E761 with ≥2 dx
   - AND at least one treatment claim
   ============================================================ */
CREATE OR REPLACE TEMPORARY VIEW specified_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e761_patients_2dx e
INNER JOIN all_tx_claims t 
    ON e.PATIENT_ID = t.PATIENT_ID;



/* ============================================================
   5) E763 PATIENTS WITH ≥2 DIAGNOSIS DATES
   ============================================================ */
CREATE OR REPLACE TEMPORARY VIEW e763_patients_2dx AS
SELECT PATIENT_ID
FROM (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)

    UNION

    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;



/* ============================================================
   6) ELAPRASE TREATMENT PATIENTS
   ============================================================ */
CREATE OR REPLACE TEMPORARY VIEW elaprase_tx AS
SELECT DISTINCT PATIENT_ID
FROM all_tx_claims
WHERE CODE IN ('54092070001', '540920700', 'J1743');



/* ============================================================
   7) INCREMENTAL PATIENTS
   - E763 with ≥2 dx
   - AND Elaprase treatment
   - NOT already in specified_patients
   ============================================================ */
CREATE OR REPLACE TEMPORARY VIEW incremental_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e763_patients_2dx e
INNER JOIN elaprase_tx t 
    ON e.PATIENT_ID = t.PATIENT_ID
WHERE e.PATIENT_ID NOT IN (
    SELECT PATIENT_ID FROM specified_patients
);



/* ============================================================
   8) ELIGIBLE PATIENTS
   ============================================================ */
CREATE OR REPLACE TEMPORARY VIEW eligible_patients AS
SELECT PATIENT_ID FROM specified_patients
UNION
SELECT PATIENT_ID FROM incremental_patients;



/* ============================================================
   9) ALL CLAIMS FOR ELIGIBLE PATIENTS
   ============================================================ */
CREATE OR REPLACE TEMPORARY VIEW all_patient_claims AS

/* Diagnosis claims */
SELECT DISTINCT
    PATIENT_ID,
    NPI,
    FILL_DATE,
    claim_id,
    plan_id,
    CLAIM_SOURCE,
    TRANSACTION_STATUS
FROM all_dx_claims
WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

UNION

/* Treatment claims */
SELECT DISTINCT
    PATIENT_ID,
    NPI,
    FILL_DATE,
    claim_id,
    plan_id,
    CLAIM_SOURCE,
    TRANSACTION_STATUS
FROM all_tx_claims
WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients);

In [0]:
SELECT * FROM all_patient_claims WHERE PATIENT_ID IN ('X7FWTST5', 'V99QKD27')

In [0]:
/*
PURPOSE
- Assign a single primary HCP per eligible patient.
- Rank HCPs using specialty priority, visit volume, and recency.
- Attach HCP and associated HCO details.

BUSINESS LOGIC
1) For each patient–NPI combination:
   - Classify provider into a specialty bucket.
   - Assign a specialty priority (lower value = higher priority).
   - Count distinct visit dates (Dx + Tx combined).
   - Capture most recent visit date.
2) Rank HCPs per patient using:
   - Specialty priority (ascending)
   - Number of visits (descending)
   - Most recent visit (descending)
   - NPI (ascending tie-breaker)
3) Select the top-ranked HCP (rank = 1) per patient.
4) Attach HCP ZIP, HCP name, and HCO details.
*/


CREATE OR REPLACE TEMPORARY VIEW primary_hcp AS

/* ============================================================
   1) Build HCP-level metrics per patient
   ============================================================ */
WITH hcp_metrics AS (
    SELECT
        a.PATIENT_ID,
        a.NPI,

        /* Specialty bucket (reporting label) */
        CASE
            WHEN p.primary_specialty LIKE '%Genetic%'
              OR p.secondary_specialty LIKE '%Genetic%'
                THEN 'Geneticist'
            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%'
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
              OR p.primary_specialty LIKE '%Neurological Surgery%'
                THEN 'Psychiatry & Neurology'
            WHEN p.primary_specialty LIKE '%Pediatrics%'
                THEN 'Pediatrician'
            WHEN p.primary_specialty LIKE '%Internal Medicine%'
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%'
              OR p.secondary_specialty LIKE '%Family Medicine%'
                THEN 'PCP'
            WHEN p.primary_specialty LIKE '%Nurse Practitioner%'
              OR p.primary_specialty LIKE '%Physician Assistant%'
                THEN 'NPPA'
            WHEN a.NPI IS NULL
                THEN 'NA'
            ELSE 'Others'
        END AS SPECIALTY,

        /* Tier 1: Specialty priority (lower = higher priority) */
        CASE
            WHEN p.primary_specialty LIKE '%Genetic%'
              OR p.secondary_specialty LIKE '%Genetic%'
                THEN 1
            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%'
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
              OR p.primary_specialty LIKE '%Neurological Surgery%'
                THEN 2
            WHEN p.primary_specialty LIKE '%Pediatrics%'
                THEN 3
            WHEN p.primary_specialty LIKE '%Internal Medicine%'
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%'
              OR p.secondary_specialty LIKE '%Family Medicine%'
                THEN 4
            WHEN p.primary_specialty LIKE '%Nurse Practitioner%'
              OR p.primary_specialty LIKE '%Physician Assistant%'
                THEN 5
            WHEN a.NPI IS NULL
                THEN 7
            ELSE 6
        END AS SPECIALTY_PRIORITY,

        /* Tier 2: Total distinct visit dates */
        COUNT(DISTINCT a.FILL_DATE) AS NO_OF_VISITS,

        /* Tier 3: Most recent visit */
        MAX(a.FILL_DATE) AS MOST_RECENT_VISIT

    FROM all_patient_claims a
    LEFT JOIN com_edp_prd.com_raw.kom_providers p
        ON a.NPI = p.NPI

    GROUP BY
        a.PATIENT_ID,
        a.NPI,
        p.primary_specialty,
        p.secondary_specialty
),

/* ============================================================
   2) Rank HCPs per patient
   ============================================================ */
ranked_hcps AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY PATIENT_ID
            ORDER BY
                SPECIALTY_PRIORITY ASC,
                NO_OF_VISITS DESC,
                MOST_RECENT_VISIT DESC,
                NPI ASC
        ) AS HCP_RANK
    FROM hcp_metrics
),

/* ============================================================
   3) Attach HCP + HCO details for top-ranked HCP
   ============================================================ */
hco_addition AS (
    SELECT
        a.PATIENT_ID AS patient_id,
        a.NPI AS hcp_npi,
        a.SPECIALTY AS hcp_specialty,

        -- /* ZIP preference: reference file first, else provider table */
        -- CASE 
        --     WHEN b.hcp_zip IS NULL OR b.hcp_zip = '-' THEN c.PROVIDER_ZIP
        --     ELSE b.hcp_zip 
        -- END AS hcp_zip,

        /* Name preference: reference file first, else provider table */
        CASE
            WHEN b.hcp_name IS NOT NULL
                THEN b.hcp_name
            ELSE CONCAT(c.FIRST_NAME, " ", c.LAST_NAME)
        END AS hcp_name,

        b.hco_veeva_crm_id,
        b.hco_name

    FROM ranked_hcps a
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.reference_file b
        ON a.NPI = b.hcp_npi
    LEFT JOIN com_raw.kom_providers c
        ON a.NPI = c.NPI
       AND c.PROVIDER_TYPE = 'INDIVIDUAL'

    WHERE a.HCP_RANK = 1
)

/* ============================================================
   Final Output: One primary HCP per patient
   ============================================================ */
SELECT DISTINCT *
FROM hco_addition;

In [0]:
SELECT
  a.patient_id,
  a.hcp_npi,
  a.hcp_specialty,
  a.hcp_name,
  a.hco_veeva_crm_id,
  a.hco_name,

  LPAD(COALESCE(
    NULLIF(b.hco_zip, '-'),
    NULLIF(b.hcp_zip, '-'),
    c.PROVIDER_ZIP
  ), 5, '0') AS final_zip,

  COALESCE(CAST(t.territory_id AS STRING), 'Unknown') AS territory_id,
  COALESCE(t.territory_name, 'Unknown')               AS territory,
  t.region_id,
  COALESCE(t.region_name, 'Unknown')                  AS region

FROM primary_hcp a

LEFT JOIN cmpa_insights_internal_schema.reference_file b
  ON a.hcp_npi = b.hcp_npi

LEFT JOIN com_raw.kom_providers c
  ON a.hcp_npi = c.NPI
 AND c.PROVIDER_TYPE = 'INDIVIDUAL'

LEFT JOIN cmpa_insights_internal_schema.zip_to_territory_mapping t
  ON LPAD(
       COALESCE(NULLIF(b.hco_zip, '-'), NULLIF(b.hcp_zip, '-'), c.PROVIDER_ZIP),
     5, '0') = LPAD(TRY_CAST(t.zipcode AS STRING), 5, '0');

In [0]:
-- ============================================================
-- QC 1: One Primary HCP per Patient
-- ============================================================

/*
These 2 numbers must match.
If not >>> multiple rank=1 rows per patient (RANK issue)
*/

SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT patient_id) AS distinct_patients
FROM primary_hcp;

In [0]:
CREATE OR REPLACE TEMP VIEW avlayah_u17_first_date AS
SELECT
    t.PATIENT_ID,
    MIN(t.FILL_DATE) AS FIRST_AVLAYAH_U17_DATE
FROM all_tx_claims t
INNER JOIN com_edp_prd.cmpa_insights_internal_schema.patient360_master p
    ON t.PATIENT_ID = p.PATIENT_ID
WHERE t.CODE IN ('8497600101')   -- 👈 Avlayah code (confirm if more)
  AND p.patient_age < 17
GROUP BY t.PATIENT_ID;

CREATE OR REPLACE TEMP VIEW new_patient_flags AS
SELECT
    a.PATIENT_ID,

    /* Overall first event */
    MIN(a.FILL_DATE) AS FIRST_EVENT_DATE,

    /* Avlayah U17 first event */
    u.FIRST_AVLAYAH_U17_DATE,

    /* Existing flags */
    CASE 
        WHEN MIN(a.FILL_DATE) >= DATEADD(month, -1, (SELECT end_date FROM runtime_parameters))
        THEN 1 ELSE 0 
    END AS NEW_PATIENT_R1M,

    CASE 
        WHEN MIN(a.FILL_DATE) >= DATEADD(month, -3, (SELECT end_date FROM runtime_parameters))
        THEN 1 ELSE 0 
    END AS NEW_PATIENT_R3M,

    /* ✅ Avlayah U17 flags */
    CASE 
        WHEN u.FIRST_AVLAYAH_U17_DATE >= DATEADD(month, -1, (SELECT end_date FROM runtime_parameters))
        THEN 1 ELSE 0 
    END AS NEW_PATIENT_R1M_AVLAYAH_U17,

    CASE 
        WHEN u.FIRST_AVLAYAH_U17_DATE >= DATEADD(month, -3, (SELECT end_date FROM runtime_parameters))
        THEN 1 ELSE 0 
    END AS NEW_PATIENT_R3M_AVLAYAH_U17

FROM all_patient_claims a
LEFT JOIN avlayah_u17_first_date u
    ON a.PATIENT_ID = u.PATIENT_ID
GROUP BY a.PATIENT_ID, u.FIRST_AVLAYAH_U17_DATE;  

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level AS

/* ============================================================
   1) Eligible patient universe
   ============================================================ */
WITH eligible_patient_universe AS (
  SELECT DISTINCT patient_id
  FROM eligible_patients
),

/* ============================================================
   2) Count distinct claims per patient
   ============================================================ */
patient_claim_counts AS (
  SELECT 
    patient_id, 
    COUNT(DISTINCT claim_id) AS claims_count
  FROM all_patient_claims
  GROUP BY 1
),

/* ============================================================
   3) Attach claims count to eligible patients
   ============================================================ */
eligible_patients_with_claims AS (
  SELECT DISTINCT 
    a.patient_id, 
    b.claims_count
  FROM eligible_patient_universe a 
  LEFT JOIN patient_claim_counts b 
    ON a.patient_id = b.patient_id
),

/* ============================================================
   4) Attach primary HCP details
   ============================================================ */
patients_with_primary_hcp AS (
  SELECT 
    a.*, 
    b.* EXCEPT (b.patient_id)
  FROM eligible_patients_with_claims a
  LEFT JOIN primary_hcp b 
    ON a.patient_id = b.patient_id
),

/* ============================================================
   5) Enrich HCP ZIP using reference file with fallback chain:
      reference_file.hco_zip → reference_file.hcp_zip → kom_providers ZIP
      ('-' values in reference file treated as NULL
   ============================================================ */
patients_with_final_zip AS (
  SELECT
    a.*,
    LPAD(COALESCE(
      NULLIF(b.hco_zip, '-'),
      NULLIF(b.hcp_zip, '-'),
      c.PROVIDER_ZIP
    ), 5, '0') AS final_zip
  FROM patients_with_primary_hcp a
  LEFT JOIN cmpa_insights_internal_schema.reference_file b
    ON a.hcp_npi = b.hcp_npi
  LEFT JOIN com_raw.kom_providers c
    ON a.hcp_npi = c.NPI
   AND c.PROVIDER_TYPE = 'INDIVIDUAL'
),

/* ============================================================
   6) Map HCP ZIP to territory and region
   ============================================================ */
patients_with_territory_region AS (
  SELECT 
    a.*, 
    b.territory_id, 
    b.territory_name AS territory, 
    b.region_id, 
    b.region_name AS region
  FROM patients_with_final_zip a
  LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping b 
    ON a.final_zip = LPAD(TRY_CAST(b.zipcode AS STRING), 5, '0')
),

/* ============================================================
   7) Determine latest plan per patient
      (Most recent claim by fill_date; tie-break by NPI)
   ============================================================ */
latest_plan_per_patient AS (
  SELECT 
    b.patient_id, 
    b.plan_id
  FROM ( 
    SELECT 
      a.patient_id, 
      a.plan_id, 
      ROW_NUMBER() OVER (
        PARTITION BY a.patient_id 
        ORDER BY a.fill_date DESC, a.npi ASC
      ) AS rn
    FROM all_patient_claims a
    WHERE a.plan_id IS NOT NULL
  ) b
  WHERE b.rn = 1
),

/* ============================================================
   8) Attach latest plan to patient record
   ============================================================ */
patients_with_latest_plan AS (
  SELECT 
    a.*, 
    b.plan_id
  FROM patients_with_territory_region a
  LEFT JOIN latest_plan_per_patient b 
    ON a.patient_id = b.patient_id
),

/* ============================================================
   9) Attach payer attributes from plan table
   ============================================================ */
patients_with_payer_attributes AS (
  SELECT DISTINCT
    a.*,
    COALESCE(r.canonical_payer_id, b.PAYER_ID, 0) AS PAYER_ID,
    COALESCE(r.canonical_payer_name, b.PAYER_NAME, 'Unknown') AS PAYER_NAME,
    COALESCE(r.canonical_payer_id, b.PAYER_ID, 0) AS PARENT_ID,
    COALESCE(r.canonical_payer_name, b.PAYER_NAME, 'Unknown') AS PARENT_NAME,
    b.INSURANCE_SEGMENT,
    b.INSURANCE_GROUP
  FROM patients_with_latest_plan a
  LEFT JOIN com_edp_prd.com_raw.kom_plans b
    ON a.plan_id = b.KH_PLAN_ID
  LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.payer_rollup_dim r
    ON b.PAYER_ID = r.source_payer_id
),

/* ============================================================
   10) Add Avlayah / Elaprase patient flags
   ============================================================ */
patients_with_tx_flags AS (
  SELECT
    p.*,
    CASE
      WHEN p360.patient_age < 17
       AND UPPER(p360.latest_mpsii_tx_type) = 'AVLAYAH'
      THEN 1
      ELSE 0
    END AS avlayah_pt_lt_17,
    CASE
      WHEN p360.patient_age < 17
       AND UPPER(p360.latest_mpsii_tx_type) IN ('ELAPRASE', 'OTHER ERT PROC')
      THEN 1
      ELSE 0
    END AS elaprase_pt_lt_17
  FROM patients_with_payer_attributes p
  LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.patient360_master p360
    ON p.patient_id = p360.patient_id
)

/* ============================================================
   Final Patient-Level Output
   ============================================================ */
SELECT
  patient_id,
  claims_count,

  hcp_npi,
  hcp_specialty,
  final_zip,        -- rename final_zip to hcp_zip for consistency
  hcp_name,
  hco_veeva_crm_id,
  hco_name,

  COALESCE(CAST(territory_id AS STRING), 'Unknown') AS territory_id,
  COALESCE(territory, 'Unknown') AS territory,

  region_id,
  COALESCE(region, 'Unknown') AS region,

  plan_id,
  COALESCE(PAYER_ID, 'Unknown') AS PAYER_ID,
  COALESCE(PAYER_NAME, 'Unknown') AS PAYER_NAME,
  COALESCE(PARENT_ID, 'Unknown') AS PARENT_ID,
  COALESCE(PARENT_NAME, 'Unknown') AS PARENT_NAME,
  COALESCE(INSURANCE_SEGMENT, 'Unknown') AS INSURANCE_SEGMENT,
  COALESCE(INSURANCE_GROUP, 'Unknown') AS INSURANCE_GROUP,

  avlayah_pt_lt_17,
  elaprase_pt_lt_17

FROM patients_with_tx_flags;

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level

In [0]:

CREATE OR REPLACE TEMP VIEW elaprase_provider_universe AS

WITH raw_provider_claims AS (

    -- ===============================
    -- MEDICAL CLAIMS (DX + NDC + PROC)
    -- ===============================
    SELECT DISTINCT
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS HCP_NPI,
        BILLING_NPI AS HCO_NPI
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE (
            DIAGNOSIS_CODES LIKE '%E761%'
         OR DIAGNOSIS_CODES LIKE '%E763%'
         OR NDC11 IN ('54092070001','540920700')
         OR PROCEDURE_CODE IN (
             '99601','99602','96365','96366','J1743',
             'S9357','S9379','38206','38230','38232',
             '38240','38241','38242','38243','38250'
         )
    )
    AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

    UNION

    -- ===============================
    -- PHARMACY CLAIMS
    -- ===============================
    SELECT DISTINCT
        PATIENT_ID,
        PRESCRIBER_NPI AS HCP_NPI,
        PHARMACY_NPI AS HCO_NPI
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE (
            DIAGNOSIS_CODE IN ('E761','E763')
         OR NDC11 IN ('54092070001','540920700')
    )
    AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
)

-- ===============================
-- Attach Payer + Territory from MASTER
-- ===============================
SELECT DISTINCT
    rpc.PATIENT_ID,
    rpc.HCP_NPI,
    rpc.HCO_NPI,

    /* canonical payer from patient master */
    COALESCE(p.PAYER_ID,0) AS payer_id,
    COALESCE(p.PAYER_NAME,'Unknown') AS payer_name,

    COALESCE(p.PARENT_ID,0) AS parent_id,
    COALESCE(p.PARENT_NAME,'Unknown') AS parent_name,

    p.territory_id,
    p.territory,
    p.region_id,
    p.region

FROM raw_provider_claims rpc

JOIN com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level p
    ON rpc.PATIENT_ID = p.patient_id;

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.payer360_master AS

/* ============================================================
1. NEW PATIENT FLAGS
============================================================ */
-- WITH new_patient_flags AS (

-- SELECT
--     patient_id,
--     MIN(fill_date) FIRST_EVENT_DATE,

--     CASE WHEN MIN(fill_date) >= DATEADD(month,-1,(SELECT end_date FROM runtime_parameters))
--         THEN 1 ELSE 0 END NEW_PATIENT_R1M,

--     CASE WHEN MIN(fill_date) >= DATEADD(month,-3,(SELECT end_date FROM runtime_parameters))
--         THEN 1 ELSE 0 END NEW_PATIENT_R3M

-- FROM all_patient_claims
-- GROUP BY patient_id
-- ),

/* ============================================================
1. NEW PATIENT FLAGS (UPDATED WITH AVLAYAH U17)
============================================================ */
WITH tx_classification AS (

SELECT 
    patient_id,
    claim_id,
    drug_type,
    is_denial,
    fill_date 
FROM (

    /* ================= MEDICAL ================= */
    SELECT 
        patient_id,
        medical_event_id AS claim_id,
        service_date AS fill_date,   -- ✅ bring it here

        CASE 
            WHEN COALESCE(ndc11, procedure_code) IN 
                ('54092070001','540920700','99601','99602','96365','96366','J1743','S9357','S9379',
                 '38206','38230','38232','38240','38241','38242','38243','38250') 
            THEN 'ELAPRASE'

            WHEN COALESCE(ndc11, procedure_code) IN 
                ('8479600101','J3490','J3590','J9999') 
                AND service_date >= '2026-03-01' 
            THEN 'AVLAYAH'
        END AS drug_type,

        0 AS is_denial

    FROM com_edp_prd.com_raw.kom_medical_events


    UNION ALL


    /* ================= PHARMACY ================= */
    SELECT 
        patient_id,
        pharmacy_event_id AS claim_id,
        fill_date,   -- ✅ already exists here

        CASE 
            WHEN ndc11 IN ('54092070001','540920700') THEN 'ELAPRASE'
            WHEN ndc11 IN ('8479600101') THEN 'AVLAYAH'
        END AS drug_type,

        CASE WHEN UPPER(transaction_result)='REJECTED' THEN 1 ELSE 0 END

    FROM com_edp_prd.com_raw.kom_pharmacy_events

) t

WHERE drug_type IS NOT NULL
),

patient_age AS (

SELECT
    p.patient_id,
    YEAR(CURRENT_DATE)-YEAR(d.patient_yob) CURRENT_AGE

FROM com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level p

LEFT JOIN (
    SELECT patient_id, MAX(patient_yob) patient_yob
    FROM com_edp_prd.com_raw.kom_patient_demographics
    GROUP BY patient_id
) d
ON p.patient_id=d.patient_id
),

new_patient_flags AS (

SELECT
    a.patient_id,

    /* Overall first event */
    MIN(a.fill_date) AS first_event_date,

    /* Avlayah U17 first event */
    MIN(CASE 
            WHEN t.drug_type = 'AVLAYAH' AND pa.current_age < 17
            THEN t.fill_date 
        END) AS first_avlayah_u17_date,

    /* ================= OVERALL FLAGS ================= */
    CASE 
        WHEN MIN(a.fill_date) >= DATEADD(month,-1,(SELECT end_date FROM runtime_parameters))
        THEN 1 ELSE 0 
    END AS new_patient_r1m,

    CASE 
        WHEN MIN(a.fill_date) >= DATEADD(month,-3,(SELECT end_date FROM runtime_parameters))
        THEN 1 ELSE 0 
    END AS new_patient_r3m,

    /* ================= AVLAYAH U17 FLAGS ================= */
    CASE 
        WHEN MIN(CASE 
                    WHEN t.drug_type = 'AVLAYAH' AND pa.current_age < 17
                    THEN t.fill_date 
                 END)
             >= DATEADD(month,-1,(SELECT end_date FROM runtime_parameters))
        THEN 1 ELSE 0 
    END AS new_patient_r1m_avlayah_u17,

    CASE 
        WHEN MIN(CASE 
                    WHEN t.drug_type = 'AVLAYAH' AND pa.current_age < 17
                    THEN t.fill_date 
                 END)
             >= DATEADD(month,-3,(SELECT end_date FROM runtime_parameters))
        THEN 1 ELSE 0 
    END AS new_patient_r3m_avlayah_u17

FROM all_patient_claims a

LEFT JOIN tx_classification t 
    ON a.patient_id = t.patient_id

LEFT JOIN patient_age pa
    ON a.patient_id = pa.patient_id

GROUP BY a.patient_id
),

/* ============================================================
2. CLAIM METRICS
============================================================ */
patient_claim_metrics AS (

SELECT
    p.patient_id,

    COUNT(DISTINCT a.claim_id) TOTAL_CLAIMS,
    COUNT(DISTINCT ph.PHARMACY_EVENT_ID) PHARMACY_TOTAL_CLAIMS,

    COUNT(DISTINCT CASE WHEN UPPER(ph.TRANSACTION_RESULT)='PAID'
        THEN ph.PHARMACY_EVENT_ID END) APPROVED_FILLS,

    COUNT(DISTINCT CASE WHEN UPPER(ph.TRANSACTION_RESULT)='REJECTED'
        THEN ph.PHARMACY_EVENT_ID END) REJECTED_FILLS,

    COUNT(DISTINCT CASE WHEN UPPER(ph.TRANSACTION_RESULT)='REVERSED'
        THEN ph.PHARMACY_EVENT_ID END) REVERSED_FILLS

FROM com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level p

LEFT JOIN (
    SELECT patient_id, claim_id
    FROM all_patient_claims
) a
ON p.patient_id=a.patient_id

LEFT JOIN com_edp_prd.com_raw.kom_pharmacy_events ph
    ON p.patient_id=ph.patient_id
   AND ph.fill_date BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
   AND (
        ph.diagnosis_code IN ('E761','E763')
        OR ph.ndc11 IN ('54092070001','540920700')
       )

GROUP BY p.patient_id
),

/* ============================================================
3. AGE
============================================================ */


payer_group_map AS (

SELECT
    p.*,

    /* payer is already canonical in patient master */
    p.payer_id AS payer_display_id,
    p.payer_name AS payer_display_name,

    /* group name equals payer name */
    p.payer_name AS payer_group

FROM com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level p
),

patient_drug_metrics AS (

SELECT 
    p.patient_id,

    /* age flag */
    CASE WHEN a.current_age < 17 THEN 1 ELSE 0 END AS is_lt17,

    /* ================= PATIENT FLAGS ================= */
    MAX(CASE WHEN t.drug_type='ELAPRASE' THEN 1 ELSE 0 END) AS elaprase_flag,
    MAX(CASE WHEN t.drug_type='AVLAYAH' THEN 1 ELSE 0 END) AS avlayah_flag,

    /* ================= CLAIM COUNTS ================= */
    COUNT(DISTINCT CASE WHEN t.drug_type='ELAPRASE' THEN t.claim_id END) AS elaprase_claims_ct,
    COUNT(DISTINCT CASE WHEN t.drug_type='AVLAYAH' THEN t.claim_id END) AS avlayah_claims_ct,

    /* ================= DENIALS ================= */
    SUM(CASE WHEN t.drug_type='ELAPRASE' AND t.is_denial=1 THEN 1 ELSE 0 END) AS elaprase_denials,
    SUM(CASE WHEN t.drug_type='AVLAYAH' AND t.is_denial=1 THEN 1 ELSE 0 END) AS avlayah_denials,

    /* ================= PRIMARY HCP / HCO ================= */
    count(CASE WHEN t.drug_type='ELAPRASE' THEN p.hcp_npi END) AS elaprase_hcp,
    count(CASE WHEN t.drug_type='AVLAYAH' THEN p.hcp_npi END) AS avlayah_hcp,

    count(CASE WHEN t.drug_type='ELAPRASE' THEN p.hco_veeva_crm_id END) AS elaprase_hco,
    count(CASE WHEN t.drug_type='AVLAYAH' THEN p.hco_veeva_crm_id END) AS avlayah_hco,

    /* ================= PAYER SPLIT (PATIENT LEVEL) ================= */
    MAX(CASE WHEN t.drug_type='ELAPRASE' AND p.insurance_group='COMMERCIAL' THEN 1 ELSE 0 END) elaprase_comm,
    MAX(CASE WHEN t.drug_type='ELAPRASE' AND p.insurance_group='MEDICARE' THEN 1 ELSE 0 END) elaprase_medicare,
    MAX(CASE WHEN t.drug_type='ELAPRASE' AND p.insurance_group='MEDICAID' THEN 1 ELSE 0 END) elaprase_medicaid,
    MAX(CASE WHEN t.drug_type='ELAPRASE' AND p.insurance_group NOT IN ('COMMERCIAL','MEDICARE','MEDICAID') THEN 1 ELSE 0 END) elaprase_other,

    MAX(CASE WHEN t.drug_type='AVLAYAH' AND p.insurance_group='COMMERCIAL' THEN 1 ELSE 0 END) avlayah_comm,
    MAX(CASE WHEN t.drug_type='AVLAYAH' AND p.insurance_group='MEDICARE' THEN 1 ELSE 0 END) avlayah_medicare,
    MAX(CASE WHEN t.drug_type='AVLAYAH' AND p.insurance_group='MEDICAID' THEN 1 ELSE 0 END) avlayah_medicaid,
    MAX(CASE WHEN t.drug_type='AVLAYAH' AND p.insurance_group NOT IN ('COMMERCIAL','MEDICARE','MEDICAID') THEN 1 ELSE 0 END) avlayah_other,

    /* ================= LT 17 FLAGS ================= */
    MAX(CASE WHEN t.drug_type='ELAPRASE' AND a.current_age < 17 THEN 1 ELSE 0 END) AS elaprase_pt_lt_17,
    MAX(CASE WHEN t.drug_type='AVLAYAH' AND a.current_age < 17 THEN 1 ELSE 0 END) AS avlayah_pt_lt_17,

    /* ================= LT17 CLAIMS ================= */
    COUNT(DISTINCT CASE 
        WHEN t.drug_type='ELAPRASE' AND a.current_age < 17 
        THEN t.claim_id END) AS elaprase_claims_lt_17,

    COUNT(DISTINCT CASE 
        WHEN t.drug_type='AVLAYAH' AND a.current_age < 17 
        THEN t.claim_id END) AS avlayah_claims_lt_17,

    /* ================= LT17 DENIALS ================= */
    SUM(CASE 
        WHEN t.drug_type='ELAPRASE' AND t.is_denial=1 AND a.current_age < 17 
        THEN 1 ELSE 0 END) AS elaprase_denial_lt_17,

    SUM(CASE 
        WHEN t.drug_type='AVLAYAH' AND t.is_denial=1 AND a.current_age < 17 
        THEN 1 ELSE 0 END) AS avlayah_denial_lt_17,

    /* ================= ELAPRASE AGE SPLIT ================= */
    MAX(CASE WHEN t.drug_type='ELAPRASE' AND a.current_age < 5 THEN 1 ELSE 0 END) AS ELAPRASE_AGE_LT_5_YRS,
    MAX(CASE WHEN t.drug_type='ELAPRASE' AND a.current_age BETWEEN 5 AND 10 THEN 1 ELSE 0 END) AS ELAPRASE_AGE_5_TO_10_YRS,
    MAX(CASE WHEN t.drug_type='ELAPRASE' AND a.current_age BETWEEN 11 AND 16 THEN 1 ELSE 0 END) AS ELAPRASE_AGE_11_TO_16_YRS,
    MAX(CASE WHEN t.drug_type='ELAPRASE' AND a.current_age >= 17 THEN 1 ELSE 0 END) AS ELAPRASE_AGE_17NGRT_YRS,

    /* ================= AVLAYAH AGE SPLIT ================= */
    MAX(CASE WHEN t.drug_type='AVLAYAH' AND a.current_age < 5 THEN 1 ELSE 0 END) AS AVLAYAH_AGE_LT_5_YRS,
    MAX(CASE WHEN t.drug_type='AVLAYAH' AND a.current_age BETWEEN 5 AND 10 THEN 1 ELSE 0 END) AS AVLAYAH_AGE_5_TO_10_YRS,
    MAX(CASE WHEN t.drug_type='AVLAYAH' AND a.current_age BETWEEN 11 AND 16 THEN 1 ELSE 0 END) AS AVLAYAH_AGE_11_TO_16_YRS,
    MAX(CASE WHEN t.drug_type='AVLAYAH' AND a.current_age >= 17 THEN 1 ELSE 0 END) AS AVLAYAH_AGE_17NGRT_YRS

FROM payer_group_map p
LEFT JOIN tx_classification t 
    ON p.patient_id = t.patient_id
LEFT JOIN patient_age a 
    ON p.patient_id = a.patient_id

GROUP BY p.patient_id, a.current_age
), 

/* ============================================================
3B. DRUG METRICS (ELAPRASE vs AVLAYAH)
============================================================ */

-- tx_classification AS (

-- SELECT 
--     patient_id,
--     claim_id,
    
--     CASE 
--         WHEN code IN ('54092070001','540920700','J1743') THEN 'ELAPRASE'
--         WHEN code = '8479600101' OR code IN ('J3490','J3590','J9999') THEN 'AVLAYAH'
--     END AS drug_type,

--     CASE WHEN UPPER(transaction_status)='REJECTED' THEN 1 ELSE 0 END AS is_denial

-- FROM (
--     SELECT 
--         patient_id,
--         medical_event_id AS claim_id,
--         COALESCE(ndc11, procedure_code) AS code,
--         'PAID' AS transaction_status
--     FROM com_edp_prd.com_raw.kom_medical_events

--     UNION ALL

--     SELECT 
--         patient_id,
--         pharmacy_event_id,
--         ndc11,
--         transaction_result
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
-- )

-- WHERE code IS NOT NULL
-- ),

-- patient_drug_metrics AS (

-- SELECT 
--     p.patient_id,

--     /* ================= <17 ================= */
--     CASE WHEN a.current_age < 17 THEN 1 ELSE 0 END AS is_lt17,

--     MAX(CASE WHEN t.drug_type='ELAPRASE' THEN 1 ELSE 0 END) elaprase_flag,
--     MAX(CASE WHEN t.drug_type='AVLAYAH' THEN 1 ELSE 0 END) avlayah_flag,

--     COUNT(DISTINCT CASE WHEN t.drug_type='ELAPRASE' THEN t.claim_id END) elaprase_claims_ct,
--     COUNT(DISTINCT CASE WHEN t.drug_type='AVLAYAH' THEN t.claim_id END) avlayah_claims_ct,

--     SUM(CASE WHEN t.drug_type='ELAPRASE' AND t.is_denial=1 THEN 1 ELSE 0 END) elaprase_denials,
--     SUM(CASE WHEN t.drug_type='AVLAYAH' AND t.is_denial=1 THEN 1 ELSE 0 END) avlayah_denials,

--     COUNT(DISTINCT CASE WHEN t.drug_type='ELAPRASE' THEN p.hcp_npi END) elaprase_hcp,
--     COUNT(DISTINCT CASE WHEN t.drug_type='AVLAYAH' THEN p.hcp_npi END) avlayah_hcp,

--     COUNT(DISTINCT CASE WHEN t.drug_type='ELAPRASE' THEN p.hco_veeva_crm_id END) elaprase_hco,
--     COUNT(DISTINCT CASE WHEN t.drug_type='AVLAYAH' THEN p.hco_veeva_crm_id END) avlayah_hco,

--     /* payer splits */
--     MAX(CASE WHEN t.drug_type='ELAPRASE' AND insurance_group='COMMERCIAL' THEN 1 ELSE 0 END) elaprase_comm,
--     MAX(CASE WHEN t.drug_type='ELAPRASE' AND insurance_group='MEDICARE' THEN 1 ELSE 0 END) elaprase_medicare,

--     MAX(CASE WHEN t.drug_type='AVLAYAH' AND insurance_group='COMMERCIAL' THEN 1 ELSE 0 END) avlayah_comm,
--     MAX(CASE WHEN t.drug_type='AVLAYAH' AND insurance_group='MEDICARE' THEN 1 ELSE 0 END) avlayah_medicare

-- FROM payer_group_map p
-- LEFT JOIN tx_classification t ON p.patient_id=t.patient_id
-- LEFT JOIN patient_age a ON p.patient_id=a.patient_id

-- GROUP BY p.patient_id, a.current_age
-- ),

/* ============================================================
4. DISPLAY PAYER ROLLUP
============================================================ */


/* ============================================================
5. PAYER INFO NORMALIZATION (SAFE BCBS MATCH)
============================================================ */
payer_info_normalized AS (

SELECT

CASE

/* =========================
UHC / OPTUM
========================= */
WHEN UPPER(payer_account_name) RLIKE 'UNITED|OPTUM'
THEN 'UHC/Optum'

/* =========================
AETNA / CVS
========================= */
WHEN UPPER(payer_account_name) RLIKE 'AETNA|CVS'
THEN 'Aetna/CVS'

/* =========================
CIGNA / ESI
========================= */
WHEN UPPER(payer_account_name) RLIKE 'CIGNA|ESI|EVERNORTH|EXPRESS SCRIPTS'
THEN 'Cigna/ESI'

/* =========================
ELEVANCE / CARELON
========================= */
WHEN UPPER(payer_account_name) RLIKE 'ANTHEM|ELEVANCE|CARELON|AMERIGROUP|WELLPOINT|EMPIRE.*BLUE'
THEN 'Elevance/Carelon'

/* =========================
HCSC PLANS (BCBS + STATE)
========================= */
WHEN UPPER(payer_account_name) RLIKE 'BLUE.*ILLINOIS|BCBS.*ILLINOIS'
THEN 'Prime Therapeutics / HCSC'

WHEN UPPER(payer_account_name) RLIKE 'BLUE.*TEXAS|BCBS.*TEXAS'
THEN 'Prime Therapeutics / HCSC'

WHEN UPPER(payer_account_name) RLIKE 'BLUE.*OKLAHOMA|BCBS.*OKLAHOMA'
THEN 'Prime Therapeutics / HCSC'

WHEN UPPER(payer_account_name) RLIKE 'BLUE.*NEW MEXICO|BCBS.*NEW MEXICO'
THEN 'Prime Therapeutics / HCSC'

/* =========================
REGIONAL BCBS
========================= */
WHEN UPPER(payer_account_name) RLIKE 'BLUE.*NORTH CAROLINA'
THEN 'BCBS NC'

WHEN UPPER(payer_account_name) RLIKE 'BLUE.*TENNESSEE'
THEN 'BCBS TN'

WHEN UPPER(payer_account_name) RLIKE 'BLUE.*MASSACHUSETTS'
THEN 'BCBS MA'

WHEN UPPER(payer_account_name) RLIKE 'BLUE.*MINNESOTA'
THEN 'BCBS MN'

WHEN UPPER(payer_account_name) RLIKE 'BLUE.*ARIZONA'
THEN 'BCBS AZ'

WHEN UPPER(payer_account_name) RLIKE 'BLUE.*ARKANSAS'
THEN 'BCBS Arkansas'

WHEN UPPER(payer_account_name) RLIKE 'BLUE.*KANSAS CITY'
THEN 'BCBS Kansas City'

WHEN UPPER(payer_account_name) RLIKE 'BLUE.*RHODE ISLAND'
THEN 'BCBS RI'

/* =========================
OTHER BCBS LICENSEES
========================= */
WHEN UPPER(payer_account_name) RLIKE 'HORIZON'
THEN 'Horizon BCBS NJ'

WHEN UPPER(payer_account_name) RLIKE 'CAREFIRST'
THEN 'Carefirst BCBS (CVS)'

WHEN UPPER(payer_account_name) RLIKE 'EXCELLUS'
THEN 'Excellus'

WHEN UPPER(payer_account_name) RLIKE 'PREMERA'
THEN 'Premera'

WHEN UPPER(payer_account_name) RLIKE 'REGENCE'
THEN 'Regence'

WHEN UPPER(payer_account_name) RLIKE 'FLORIDA BLUE'
THEN 'Florida Blue'

WHEN UPPER(payer_account_name) RLIKE 'HIGHMARK'
THEN 'Highmark'

/* =========================
DEFAULT
========================= */
ELSE payer_account_name

END AS normalized_payer_display_name,

MAX(pie_completed) pie_completed,
MAX(account_director) account_director

FROM com_edp_prd.cmpa_insights_internal_schema.payer_info
GROUP BY 1

),

/* ============================================================
6. BASE ENRICHMENT
============================================================ */
base_enriched AS (

SELECT
    p.*,

    COALESCE(i.pie_completed,'NO') pie_completed,
    COALESCE(i.account_director,'-') account_director,

    COALESCE(n.new_patient_r1m,0) new_patient_r1m,
    COALESCE(n.new_patient_r3m,0) new_patient_r3m,

    COALESCE(n.new_patient_r1m_avlayah_u17,0) new_patient_r1m_avlayah_u17,
    COALESCE(n.new_patient_r3m_avlayah_u17,0) new_patient_r3m_avlayah_u17,

    COALESCE(c.total_claims,0) total_claims,
    COALESCE(c.pharmacy_total_claims,0) pharmacy_total_claims,
    COALESCE(c.approved_fills,0) approved_fills,
    COALESCE(c.rejected_fills,0) rejected_fills,
    COALESCE(c.reversed_fills,0) reversed_fills,

    COALESCE(d.is_lt17,0) is_lt17,

    COALESCE(d.elaprase_claims_ct,0) elaprase_claims_ct,
    COALESCE(d.avlayah_claims_ct,0) avlayah_claims_ct,

    COALESCE(d.elaprase_denials,0) elaprase_denials,
    COALESCE(d.avlayah_denials,0) avlayah_denials,

    COALESCE(d.elaprase_hcp,0) elaprase_hcp,
    COALESCE(d.avlayah_hcp,0) avlayah_hcp,

    COALESCE(d.elaprase_hco,0) elaprase_hco,
    COALESCE(d.avlayah_hco,0) avlayah_hco,

    COALESCE(d.elaprase_comm,0) elaprase_comm,
    COALESCE(d.elaprase_medicare,0) elaprase_medicare,
    COALESCE(d.elaprase_medicaid,0) elaprase_medicaid,
    COALESCE(d.elaprase_other,0) elaprase_other,

    COALESCE(d.avlayah_comm,0) avlayah_comm,
    COALESCE(d.avlayah_medicare,0) avlayah_medicare,
    COALESCE(d.avlayah_medicaid,0) avlayah_medicaid,
    COALESCE(d.avlayah_other,0) avlayah_other,

    COALESCE(d.elaprase_flag,0) elaprase_flag,  
    COALESCE(d.avlayah_flag,0) avlayah_flag,

    COALESCE(d.elaprase_claims_lt_17,0) elaprase_claims_lt_17,
    COALESCE(d.avlayah_claims_lt_17,0) avlayah_claims_lt_17,

    COALESCE(d.elaprase_denial_lt_17,0) elaprase_denial_lt_17,
    COALESCE(d.avlayah_denial_lt_17,0) avlayah_denial_lt_17,

    COALESCE(d.ELAPRASE_AGE_LT_5_YRS,0) ELAPRASE_AGE_LT_5_YRS,
    COALESCE(d.ELAPRASE_AGE_5_TO_10_YRS,0) ELAPRASE_AGE_5_TO_10_YRS,
    COALESCE(d.ELAPRASE_AGE_11_TO_16_YRS,0) ELAPRASE_AGE_11_TO_16_YRS,
    COALESCE(d.ELAPRASE_AGE_17NGRT_YRS,0) ELAPRASE_AGE_17NGRT_YRS,

    COALESCE(d.AVLAYAH_AGE_LT_5_YRS,0) AVLAYAH_AGE_LT_5_YRS,
    COALESCE(d.AVLAYAH_AGE_5_TO_10_YRS,0) AVLAYAH_AGE_5_TO_10_YRS,
    COALESCE(d.AVLAYAH_AGE_11_TO_16_YRS,0) AVLAYAH_AGE_11_TO_16_YRS,
    COALESCE(d.AVLAYAH_AGE_17NGRT_YRS,0) AVLAYAH_AGE_17NGRT_YRS,

    a.current_age,

    CASE WHEN a.current_age<5 THEN 1 ELSE 0 END age_lt_5_yrs,
    CASE WHEN a.current_age BETWEEN 5 AND 10 THEN 1 ELSE 0 END age_5_to_10_yrs,
    CASE WHEN a.current_age BETWEEN 11 AND 16 THEN 1 ELSE 0 END AGE_11_TO_16_YRS,
    CASE WHEN a.current_age>=17 THEN 1 ELSE 0 END age_17ngrt_yrs

FROM payer_group_map p

LEFT JOIN new_patient_flags n USING(patient_id)
LEFT JOIN patient_claim_metrics c USING(patient_id)
LEFT JOIN patient_age a USING(patient_id)
LEFT JOIN patient_drug_metrics d USING(patient_id)

LEFT JOIN payer_info_normalized i
    ON UPPER(p.payer_display_name)=UPPER(i.normalized_payer_display_name)
),

/* ============================================================
7. TERRITORY + PAYER
============================================================ */
rollup_territory_payer AS (

SELECT


territory_id,
territory AS territory_name,
CAST(payer_id AS STRING) AS payer_id,
payer_group AS payer_name,
payer_group,
CONCAT_WS(' | ',SORT_ARRAY(COLLECT_SET(parent_id))) parent_id,
CONCAT_WS(' | ',SORT_ARRAY(COLLECT_SET(parent_name))) parent_name,

COUNT(DISTINCT patient_id) total_elaprase_patients,

COUNT(DISTINCT CASE WHEN insurance_group='MEDICARE' THEN patient_id END) medicare_patients,
COUNT(DISTINCT CASE WHEN insurance_group='MEDICAID' THEN patient_id END) medicaid_patients,
COUNT(DISTINCT CASE WHEN insurance_group='COMMERCIAL' THEN patient_id END) commercial_patients,

COUNT(DISTINCT  patient_id ) elaprase_total_patients,
-- COUNT(DISTINCT CASE WHEN elaprase_flag=1 THEN patient_id END) elaprase_pt_total,
COUNT(DISTINCT CASE WHEN avlayah_flag=1 THEN patient_id END) avlayah_pt_total,
SUM(total_claims) total_claims_lt17,
SUM(elaprase_claims_ct) elaprase_claims_ct,
SUM(avlayah_claims_ct ) avlayah_claims_ct,
COUNT(DISTINCT CASE WHEN elaprase_flag=1 THEN hcp_npi END) elaprase_hcp,
COUNT(DISTINCT CASE WHEN avlayah_flag=1 THEN hcp_npi END) avlayah_hcp,
COUNT(DISTINCT CASE WHEN elaprase_flag=1 THEN hco_veeva_crm_id END) elaprase_hco,
COUNT(DISTINCT CASE WHEN avlayah_flag=1 THEN hco_veeva_crm_id END) avlayah_hco,

SUM(elaprase_comm) elaprase_commercial_pt_ct,
SUM(elaprase_medicare) elaprase_medicare_pt_ct,
SUM(elaprase_medicaid) elaprase_medicaid_pt_ct,
SUM(elaprase_other) elaprase_other_pt_ct,

SUM(avlayah_comm) avlayah_commercial_pt_ct,
SUM(avlayah_medicare) avlayah_medicare_pt_ct,
SUM(avlayah_medicaid) avlayah_medicaid_pt_ct,
SUM(avlayah_other) avlayah_other_pt_ct,
SUM(elaprase_denials) elaprase_denials,
SUM(avlayah_denials) avlayah_denials,

SUM(elaprase_claims_lt_17) elaprase_claims_lt_17,
SUM(avlayah_claims_lt_17) avlayah_claims_lt_17,

CASE 
    WHEN SUM(elaprase_claims_lt_17) = 0 THEN 0
    ELSE ROUND(100.0 * SUM(elaprase_denial_lt_17) / SUM(elaprase_claims_lt_17), 2)
END AS elaprase_denial_lt_17,

CASE 
    WHEN SUM(avlayah_claims_lt_17) = 0 THEN 0
    ELSE ROUND(100.0 * SUM(avlayah_denial_lt_17) / SUM(avlayah_claims_lt_17), 2)
END AS avlayah_denial_lt_17,

SUM(ELAPRASE_AGE_LT_5_YRS) ELAPRASE_AGE_LT_5_YRS,
SUM(ELAPRASE_AGE_5_TO_10_YRS) ELAPRASE_AGE_5_TO_10_YRS,
SUM(ELAPRASE_AGE_11_TO_16_YRS) ELAPRASE_AGE_11_TO_16_YRS,
SUM(ELAPRASE_AGE_17NGRT_YRS) ELAPRASE_AGE_17NGRT_YRS,

SUM(AVLAYAH_AGE_LT_5_YRS) AVLAYAH_AGE_LT_5_YRS,
SUM(AVLAYAH_AGE_5_TO_10_YRS) AVLAYAH_AGE_5_TO_10_YRS,
SUM(AVLAYAH_AGE_11_TO_16_YRS) AVLAYAH_AGE_11_TO_16_YRS,
SUM(AVLAYAH_AGE_17NGRT_YRS) AVLAYAH_AGE_17NGRT_YRS,

COUNT(DISTINCT CASE WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
    OR insurance_group IS NULL THEN patient_id END) other_patients,

SUM(new_patient_r1m) new_elaprase_patients_r1m,
SUM(new_patient_r3m) new_elaprase_patients_r3m,

SUM(new_patient_r1m_avlayah_u17) new_avlayah_u17_patients_r1m,
SUM(new_patient_r3m_avlayah_u17) new_avlayah_u17_patients_r3m,

COUNT(DISTINCT CASE WHEN age_lt_5_yrs=1 THEN patient_id END) age_lt_5_yrs,
COUNT(DISTINCT CASE WHEN age_5_to_10_yrs=1 THEN patient_id END) age_5_to_10_yrs,
COUNT(DISTINCT CASE WHEN AGE_11_TO_16_YRS=1 THEN patient_id END) AGE_11_TO_16_YRS,
COUNT(DISTINCT CASE WHEN age_17ngrt_yrs=1 THEN patient_id END) age_17ngrt_yrs,

SUM(COALESCE(avlayah_pt_lt_17,0)) avlayah_pt_lt_17,
SUM(COALESCE(elaprase_pt_lt_17,0)) elaprase_pt_lt_17,

COUNT(DISTINCT hcp_npi) total_primary_hcps,
COUNT(DISTINCT hco_veeva_crm_id) total_primary_hcos,

SUM(total_claims) total_claims,
SUM(pharmacy_total_claims) pharmacy_total_claims,
SUM(approved_fills) approved_fills,
SUM(rejected_fills) rejected_fills,
SUM(reversed_fills) reversed_fills,

MAX(pie_completed) pie_completed,
MAX(account_director) account_director,

'TERRITORY_PAYER' rollup_level

FROM base_enriched
GROUP BY territory_id,territory,payer_id,payer_group
),

/* ============================================================
8. PAYER ALL TERRITORY
============================================================ */
rollup_payer_all_territory AS (

SELECT
'ALL Territories' territory_id,
'All Territories' territory_name,
CAST(payer_id AS STRING) AS payer_id,
payer_group payer_name,
payer_group,
'ALL Parents' parent_id,
'All Parents' parent_name,

COUNT(DISTINCT patient_id) total_elaprase_patients,

COUNT(DISTINCT CASE WHEN insurance_group='MEDICARE' THEN patient_id END) medicare_patients,
COUNT(DISTINCT CASE WHEN insurance_group='MEDICAID' THEN patient_id END) medicaid_patients,
COUNT(DISTINCT CASE WHEN insurance_group='COMMERCIAL' THEN patient_id END) commercial_patients,

COUNT(DISTINCT  patient_id ) elaprase_total_patients,
-- COUNT(DISTINCT CASE WHEN elaprase_flag=1 THEN patient_id END) elaprase_pt_total,
COUNT(DISTINCT CASE WHEN avlayah_flag=1 THEN patient_id END) avlayah_pt_total,
SUM( total_claims ) total_claims_lt17,
SUM(elaprase_claims_ct) elaprase_claims_ct,
SUM(avlayah_claims_ct ) avlayah_claims_ct,
COUNT(DISTINCT CASE WHEN elaprase_flag=1 THEN hcp_npi END) elaprase_hcp,
COUNT(DISTINCT CASE WHEN avlayah_flag=1 THEN hcp_npi END) avlayah_hcp,
COUNT(DISTINCT CASE WHEN elaprase_flag=1 THEN hco_veeva_crm_id END) elaprase_hco,
COUNT(DISTINCT CASE WHEN avlayah_flag=1 THEN hco_veeva_crm_id END) avlayah_hco,

SUM(elaprase_comm) elaprase_commercial_pt_ct,
SUM(elaprase_medicare) elaprase_medicare_pt_ct,
SUM(elaprase_medicaid) elaprase_medicaid_pt_ct,
SUM(elaprase_other) elaprase_other_pt_ct,

SUM(avlayah_comm) avlayah_commercial_pt_ct,
SUM(avlayah_medicare) avlayah_medicare_pt_ct,
SUM(avlayah_medicaid) avlayah_medicaid_pt_ct,
SUM(avlayah_other) avlayah_other_pt_ct,
SUM(elaprase_denials) elaprase_denials,
SUM(avlayah_denials) avlayah_denials,

SUM(elaprase_claims_lt_17) elaprase_claims_lt_17,
SUM(avlayah_claims_lt_17) avlayah_claims_lt_17,

CASE 
    WHEN SUM(elaprase_claims_lt_17) = 0 THEN 0
    ELSE ROUND(100.0 * SUM(elaprase_denial_lt_17) / SUM(elaprase_claims_lt_17), 2)
END AS elaprase_denial_lt_17,

CASE 
    WHEN SUM(avlayah_claims_lt_17) = 0 THEN 0
    ELSE ROUND(100.0 * SUM(avlayah_denial_lt_17) / SUM(avlayah_claims_lt_17), 2)
END AS avlayah_denial_lt_17,

SUM(ELAPRASE_AGE_LT_5_YRS) ELAPRASE_AGE_LT_5_YRS,
SUM(ELAPRASE_AGE_5_TO_10_YRS) ELAPRASE_AGE_5_TO_10_YRS,
SUM(ELAPRASE_AGE_11_TO_16_YRS) ELAPRASE_AGE_11_TO_16_YRS,
SUM(ELAPRASE_AGE_17NGRT_YRS) ELAPRASE_AGE_17NGRT_YRS,

SUM(AVLAYAH_AGE_LT_5_YRS) AVLAYAH_AGE_LT_5_YRS,
SUM(AVLAYAH_AGE_5_TO_10_YRS) AVLAYAH_AGE_5_TO_10_YRS,
SUM(AVLAYAH_AGE_11_TO_16_YRS) AVLAYAH_AGE_11_TO_16_YRS,
SUM(AVLAYAH_AGE_17NGRT_YRS) AVLAYAH_AGE_17NGRT_YRS,

COUNT(DISTINCT CASE WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
    OR insurance_group IS NULL THEN patient_id END) other_patients,

SUM(new_patient_r1m) new_elaprase_patients_r1m,
SUM(new_patient_r3m) new_elaprase_patients_r3m,

SUM(new_patient_r1m_avlayah_u17) new_avlayah_u17_patients_r1m,
SUM(new_patient_r3m_avlayah_u17) new_avlayah_u17_patients_r3m,

COUNT(DISTINCT CASE WHEN age_lt_5_yrs=1 THEN patient_id END) age_lt_5_yrs,
COUNT(DISTINCT CASE WHEN age_5_to_10_yrs=1 THEN patient_id END) age_5_to_10_yrs,
COUNT(DISTINCT CASE WHEN AGE_11_TO_16_YRS=1 THEN patient_id END) AGE_11_TO_16_YRS,
COUNT(DISTINCT CASE WHEN age_17ngrt_yrs=1 THEN patient_id END) age_17ngrt_yrs,

SUM(COALESCE(avlayah_pt_lt_17,0)) avlayah_pt_lt_17,
SUM(COALESCE(elaprase_pt_lt_17,0)) elaprase_pt_lt_17,

COUNT(DISTINCT hcp_npi) total_primary_hcps,
COUNT(DISTINCT hco_veeva_crm_id) total_primary_hcos,

SUM(total_claims) total_claims,
SUM(pharmacy_total_claims) pharmacy_total_claims,
SUM(approved_fills) approved_fills,
SUM(rejected_fills) rejected_fills,
SUM(reversed_fills) reversed_fills,

MAX(pie_completed) pie_completed,
MAX(account_director) account_director,

'PAYER_ALL_TERRITORY' rollup_level

FROM base_enriched
GROUP BY payer_id,payer_group
),

/* ============================================================
9. TERRITORY ALL PAYER
============================================================ */
rollup_territory_all_payer AS (

SELECT
territory_id,
territory territory_name,
'ALL Payers' payer_id,
'All Payers' payer_name,
'All Payers' payer_group,
'ALL Parents' parent_id,
'All Parents' parent_name,

COUNT(DISTINCT patient_id) total_elaprase_patients,

COUNT(DISTINCT CASE WHEN insurance_group='MEDICARE' THEN patient_id END) medicare_patients,
COUNT(DISTINCT CASE WHEN insurance_group='MEDICAID' THEN patient_id END) medicaid_patients,
COUNT(DISTINCT CASE WHEN insurance_group='COMMERCIAL' THEN patient_id END) commercial_patients,

COUNT(DISTINCT  patient_id ) elaprase_total_patients,
-- COUNT(DISTINCT CASE WHEN elaprase_flag=1 THEN patient_id END) elaprase_pt_total,
COUNT(DISTINCT CASE WHEN avlayah_flag=1 THEN patient_id END) avlayah_pt_total,
SUM( total_claims ) total_claims_lt17,
SUM(elaprase_claims_ct) elaprase_claims_ct,
SUM(avlayah_claims_ct ) avlayah_claims_ct,
COUNT(DISTINCT CASE WHEN elaprase_flag=1 THEN hcp_npi END) elaprase_hcp,
COUNT(DISTINCT CASE WHEN avlayah_flag=1 THEN hcp_npi END) avlayah_hcp,
COUNT(DISTINCT CASE WHEN elaprase_flag=1 THEN hco_veeva_crm_id END) elaprase_hco,
COUNT(DISTINCT CASE WHEN avlayah_flag=1 THEN hco_veeva_crm_id END) avlayah_hco,

SUM(elaprase_comm) elaprase_commercial_pt_ct,
SUM(elaprase_medicare) elaprase_medicare_pt_ct,
SUM(elaprase_medicaid) elaprase_medicaid_pt_ct,
SUM(elaprase_other) elaprase_other_pt_ct,

SUM(avlayah_comm) avlayah_commercial_pt_ct,
SUM(avlayah_medicare) avlayah_medicare_pt_ct,
SUM(avlayah_medicaid) avlayah_medicaid_pt_ct,
SUM(avlayah_other) avlayah_other_pt_ct,
SUM(elaprase_denials) elaprase_denials,
SUM(avlayah_denials) avlayah_denials,

SUM(elaprase_claims_lt_17) elaprase_claims_lt_17,
SUM(avlayah_claims_lt_17) avlayah_claims_lt_17,

CASE 
    WHEN SUM(elaprase_claims_lt_17) = 0 THEN 0
    ELSE ROUND(100.0 * SUM(elaprase_denial_lt_17) / SUM(elaprase_claims_lt_17), 2)
END AS elaprase_denial_lt_17,

CASE 
    WHEN SUM(avlayah_claims_lt_17) = 0 THEN 0
    ELSE ROUND(100.0 * SUM(avlayah_denial_lt_17) / SUM(avlayah_claims_lt_17), 2)
END AS avlayah_denial_lt_17,

SUM(ELAPRASE_AGE_LT_5_YRS) ELAPRASE_AGE_LT_5_YRS,
SUM(ELAPRASE_AGE_5_TO_10_YRS) ELAPRASE_AGE_5_TO_10_YRS,
SUM(ELAPRASE_AGE_11_TO_16_YRS) ELAPRASE_AGE_11_TO_16_YRS,
SUM(ELAPRASE_AGE_17NGRT_YRS) ELAPRASE_AGE_17NGRT_YRS,

SUM(AVLAYAH_AGE_LT_5_YRS) AVLAYAH_AGE_LT_5_YRS,
SUM(AVLAYAH_AGE_5_TO_10_YRS) AVLAYAH_AGE_5_TO_10_YRS,
SUM(AVLAYAH_AGE_11_TO_16_YRS) AVLAYAH_AGE_11_TO_16_YRS,
SUM(AVLAYAH_AGE_17NGRT_YRS) AVLAYAH_AGE_17NGRT_YRS,

COUNT(DISTINCT CASE WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
    OR insurance_group IS NULL THEN patient_id END) other_patients,

SUM(new_patient_r1m) new_elaprase_patients_r1m,
SUM(new_patient_r3m) new_elaprase_patients_r3m,

SUM(new_patient_r1m_avlayah_u17) new_avlayah_u17_patients_r1m,
SUM(new_patient_r3m_avlayah_u17) new_avlayah_u17_patients_r3m,

COUNT(DISTINCT CASE WHEN age_lt_5_yrs=1 THEN patient_id END) age_lt_5_yrs,
COUNT(DISTINCT CASE WHEN age_5_to_10_yrs=1 THEN patient_id END) age_5_to_10_yrs,
COUNT(DISTINCT CASE WHEN AGE_11_TO_16_YRS=1 THEN patient_id END) AGE_11_TO_16_YRS,
COUNT(DISTINCT CASE WHEN age_17ngrt_yrs=1 THEN patient_id END) age_17ngrt_yrs,

SUM(COALESCE(avlayah_pt_lt_17,0)) avlayah_pt_lt_17,
SUM(COALESCE(elaprase_pt_lt_17,0)) elaprase_pt_lt_17,

COUNT(DISTINCT hcp_npi) total_primary_hcps,
COUNT(DISTINCT hco_veeva_crm_id) total_primary_hcos,

SUM(total_claims) total_claims,
SUM(pharmacy_total_claims) pharmacy_total_claims,
SUM(approved_fills) approved_fills,
SUM(rejected_fills) rejected_fills,
SUM(reversed_fills) reversed_fills,

CAST(NULL AS STRING) pie_completed,
CAST(NULL AS STRING) account_director,

'TERRITORY_ALL_PAYER' rollup_level

FROM base_enriched
GROUP BY territory_id,territory
),

/* ============================================================
10. NATIONAL
============================================================ */
rollup_national AS (

SELECT
'ALL Territories' territory_id,
'All Territories' territory_name,
'ALL Payers' payer_id,
'All Payers' payer_name,
'All Payers' payer_group,
'ALL Parents' parent_id,
'All Parents' parent_name,

COUNT(DISTINCT patient_id) total_elaprase_patients,

COUNT(DISTINCT CASE WHEN insurance_group='MEDICARE' THEN patient_id END) medicare_patients,
COUNT(DISTINCT CASE WHEN insurance_group='MEDICAID' THEN patient_id END) medicaid_patients,
COUNT(DISTINCT CASE WHEN insurance_group='COMMERCIAL' THEN patient_id END) commercial_patients,

COUNT(DISTINCT  patient_id ) elaprase_total_patients,
-- COUNT(DISTINCT CASE WHEN elaprase_flag=1 THEN patient_id END) elaprase_pt_total,
COUNT(DISTINCT CASE WHEN avlayah_flag=1 THEN patient_id END) avlayah_pt_total,
SUM( total_claims ) total_claims_lt17,
SUM(elaprase_claims_ct) elaprase_claims_ct,
SUM(avlayah_claims_ct ) avlayah_claims_ct,
COUNT(DISTINCT CASE WHEN elaprase_flag=1 THEN hcp_npi END) elaprase_hcp,
COUNT(DISTINCT CASE WHEN avlayah_flag=1 THEN hcp_npi END) avlayah_hcp,
COUNT(DISTINCT CASE WHEN elaprase_flag=1 THEN hco_veeva_crm_id END) elaprase_hco,
COUNT(DISTINCT CASE WHEN avlayah_flag=1 THEN hco_veeva_crm_id END) avlayah_hco,

SUM(elaprase_comm) elaprase_commercial_pt_ct,
SUM(elaprase_medicare) elaprase_medicare_pt_ct,
SUM(elaprase_medicaid) elaprase_medicaid_pt_ct,
SUM(elaprase_other) elaprase_other_pt_ct,

SUM(avlayah_comm) avlayah_commercial_pt_ct,
SUM(avlayah_medicare) avlayah_medicare_pt_ct,
SUM(avlayah_medicaid) avlayah_medicaid_pt_ct,
SUM(avlayah_other) avlayah_other_pt_ct,
SUM(elaprase_denials) elaprase_denials,
SUM(avlayah_denials) avlayah_denials,

SUM(elaprase_claims_lt_17) elaprase_claims_lt_17,
SUM(avlayah_claims_lt_17) avlayah_claims_lt_17,

CASE 
    WHEN SUM(elaprase_claims_lt_17) = 0 THEN 0
    ELSE ROUND(100.0 * SUM(elaprase_denial_lt_17) / SUM(elaprase_claims_lt_17), 2)
END AS elaprase_denial_lt_17,

CASE 
    WHEN SUM(avlayah_claims_lt_17) = 0 THEN 0
    ELSE ROUND(100.0 * SUM(avlayah_denial_lt_17) / SUM(avlayah_claims_lt_17), 2)
END AS avlayah_denial_lt_17,

SUM(ELAPRASE_AGE_LT_5_YRS) ELAPRASE_AGE_LT_5_YRS,
SUM(ELAPRASE_AGE_5_TO_10_YRS) ELAPRASE_AGE_5_TO_10_YRS,
SUM(ELAPRASE_AGE_11_TO_16_YRS) ELAPRASE_AGE_11_TO_16_YRS,
SUM(ELAPRASE_AGE_17NGRT_YRS) ELAPRASE_AGE_17NGRT_YRS,

SUM(AVLAYAH_AGE_LT_5_YRS) AVLAYAH_AGE_LT_5_YRS,
SUM(AVLAYAH_AGE_5_TO_10_YRS) AVLAYAH_AGE_5_TO_10_YRS,
SUM(AVLAYAH_AGE_11_TO_16_YRS) AVLAYAH_AGE_11_TO_16_YRS,
SUM(AVLAYAH_AGE_17NGRT_YRS) AVLAYAH_AGE_17NGRT_YRS,

COUNT(DISTINCT CASE WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
    OR insurance_group IS NULL THEN patient_id END) other_patients,

SUM(new_patient_r1m) new_elaprase_patients_r1m,
SUM(new_patient_r3m) new_elaprase_patients_r3m,

SUM(new_patient_r1m_avlayah_u17) new_avlayah_u17_patients_r1m,
SUM(new_patient_r3m_avlayah_u17) new_avlayah_u17_patients_r3m,

COUNT(DISTINCT CASE WHEN age_lt_5_yrs=1 THEN patient_id END) age_lt_5_yrs,
COUNT(DISTINCT CASE WHEN age_5_to_10_yrs=1 THEN patient_id END) age_5_to_10_yrs,
COUNT(DISTINCT CASE WHEN AGE_11_TO_16_YRS=1 THEN patient_id END) AGE_11_TO_16_YRS,
COUNT(DISTINCT CASE WHEN age_17ngrt_yrs=1 THEN patient_id END) age_17ngrt_yrs,

SUM(COALESCE(avlayah_pt_lt_17,0)) avlayah_pt_lt_17,
SUM(COALESCE(elaprase_pt_lt_17,0)) elaprase_pt_lt_17,

COUNT(DISTINCT hcp_npi) total_primary_hcps,
COUNT(DISTINCT hco_veeva_crm_id) total_primary_hcos,

SUM(total_claims) total_claims,
SUM(pharmacy_total_claims) pharmacy_total_claims,
SUM(approved_fills) approved_fills,
SUM(rejected_fills) rejected_fills,
SUM(reversed_fills) reversed_fills,

CAST(NULL AS STRING) pie_completed,
CAST(NULL AS STRING) account_director,

'NATIONAL' rollup_level

FROM base_enriched
),

/* ============================================================
11. UNION ALL ROLLUPS
============================================================ */
all_rollups AS (

SELECT * FROM rollup_territory_payer
UNION ALL
SELECT * FROM rollup_payer_all_territory
UNION ALL
SELECT * FROM rollup_territory_all_payer
UNION ALL
SELECT * FROM rollup_national

),

final_with_lives AS (

SELECT
r.*,
COALESCE(t.total_lives,0) total_lives

FROM all_rollups r

LEFT JOIN total_lives t
  ON r.rollup_level=t.rollup_level
 AND r.territory_id=t.territory_id
 AND r.payer_id=t.payer_id

),

final_with_share AS (

SELECT
f.*,

100.0 * f.total_lives /
NULLIF(
CASE
WHEN f.rollup_level='TERRITORY_PAYER'
THEN SUM(f.total_lives) OVER(PARTITION BY f.rollup_level,f.territory_id)
ELSE SUM(f.total_lives) OVER(PARTITION BY f.rollup_level)
END
,0) payer_market_share_pct

FROM final_with_lives f
),

/* ============================================================
FINAL RANK CALCULATION (EXCLUDE UNKNOWN FROM RANKING)
============================================================ */
final_with_rank AS (

WITH ranked_payers AS (

SELECT
f.*,

RANK() OVER(
PARTITION BY
CASE
    WHEN rollup_level='TERRITORY_PAYER'
        THEN territory_id
    ELSE rollup_level
END
ORDER BY total_elaprase_patients DESC
) AS payer_rank,

ROW_NUMBER() OVER(
PARTITION BY
CASE
    WHEN rollup_level='TERRITORY_PAYER'
        THEN territory_id
    ELSE rollup_level
END
ORDER BY total_elaprase_patients DESC
) AS row_num

FROM final_with_share f
WHERE payer_name NOT IN ('Unknown','All Parents')

),

rank_counts AS (

SELECT
CASE
    WHEN rollup_level='TERRITORY_PAYER'
        THEN territory_id
    ELSE rollup_level
END AS grp,
COUNT(*) AS payer_count
FROM ranked_payers
GROUP BY 1

),

special_rows AS (

SELECT
f.*,

CASE
WHEN payer_name='Unknown'
THEN r.payer_count
WHEN payer_name='All Parents'
THEN r.payer_count + 1
END AS payer_rank,

NULL AS row_num

FROM final_with_share f
JOIN rank_counts r
ON (
CASE
    WHEN f.rollup_level='TERRITORY_PAYER'
        THEN f.territory_id
    ELSE f.rollup_level
END
=
r.grp
)

WHERE payer_name IN ('Unknown','All Parents')

)

SELECT * FROM ranked_payers
UNION ALL
SELECT * FROM special_rows

)

SELECT

territory_id,
territory_name,
CAST(payer_id AS STRING) AS payer_id,
payer_name,
parent_id,
parent_name,

payer_market_share_pct,
payer_rank,
total_lives,

total_elaprase_patients,

medicare_patients,
medicaid_patients,
commercial_patients,
other_patients,

new_elaprase_patients_r1m      AS NEW_ELAPRASE_PATIENTS_R1M,
new_elaprase_patients_r3m      AS NEW_ELAPRASE_PATIENTS_R3M,

new_avlayah_u17_patients_r1m AS AVLAYAH_U17_NEW_PATIENTS_R1M,
new_avlayah_u17_patients_r3m AS AVLAYAH_U17_NEW_PATIENTS_R3M,

age_lt_5_yrs                   AS AGE_LT_5_YRS,
age_5_to_10_yrs                AS AGE_5_TO_10_YRS,
AGE_11_TO_16_YRS               AS AGE_11_TO_16_YRS,
age_17ngrt_yrs                 AS AGE_17NGRT_YRS,

avlayah_pt_lt_17   AS AVLAYAH_PT_LT_17,
elaprase_pt_lt_17  AS ELAPRASE_PT_LT_17,

total_primary_hcps             AS TOTAL_PRIMARY_HCPS,
total_primary_hcos             AS TOTAL_PRIMARY_HCOS,

total_claims                   AS TOTAL_CLAIMS,
pharmacy_total_claims          AS PHARMACY_TOTAL_CLAIMS,

approved_fills                 AS APPROVED_FILLS,
rejected_fills                 AS REJECTED_FILLS,
reversed_fills                 AS REVERSED_FILLS,

-- <17 metrics
elaprase_total_patients,
-- elaprase_pt_total,
avlayah_pt_total,
total_claims_lt17,

-- claims
elaprase_claims_ct,
avlayah_claims_ct,

-- HCP/HCO
elaprase_hcp,
avlayah_hcp,
elaprase_hco,
avlayah_hco,

-- payer split
elaprase_commercial_pt_ct,
elaprase_medicare_pt_ct,
elaprase_medicaid_pt_ct,
elaprase_other_pt_ct,

avlayah_commercial_pt_ct,
avlayah_medicare_pt_ct,
avlayah_medicaid_pt_ct,
avlayah_other_pt_ct,    

-- age
elaprase_age_lt_5_yrs,
elaprase_age_5_to_10_yrs,
elaprase_age_11_to_16_yrs,
elaprase_age_17ngrt_yrs,

avlayah_age_lt_5_yrs,
avlayah_age_5_to_10_yrs,
avlayah_age_11_to_16_yrs,
avlayah_age_17ngrt_yrs,

-- denials
elaprase_denial_lt_17,
avlayah_denial_lt_17,

elaprase_claims_lt_17,
avlayah_claims_lt_17,

-- denial rates
CASE WHEN elaprase_claims_ct=0 THEN 0
ELSE ROUND(100.0*elaprase_denials/elaprase_claims_ct,2)
END AS elaprase_denial_rate,

CASE WHEN avlayah_claims_ct=0 THEN 0
ELSE ROUND(100.0*avlayah_denials/avlayah_claims_ct,2)
END AS avlayah_denial_rate,

CASE
    WHEN pharmacy_total_claims = 0 THEN 0
    ELSE ROUND(100.0 * approved_fills / pharmacy_total_claims,2)
END AS ELAPRASE_APPROVAL_RATE,

CASE
    WHEN pharmacy_total_claims = 0 THEN 0
    ELSE ROUND(100.0 * rejected_fills / pharmacy_total_claims,2)
END AS ELAPRASE_REJECTION_RATE,

CASE
    WHEN pharmacy_total_claims = 0 THEN 0
    ELSE ROUND(100.0 * reversed_fills / pharmacy_total_claims,2)
END AS ELAPRASE_REVERSED_RATE,

pie_completed                  AS PIE_COMPLETED,
account_director               AS ACCOUNT_DIRECTOR,

rollup_level

FROM final_with_rank;

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.payer360_master

In [0]:
SELECT * FROM com_edp_prd.cmpa_insights_internal_schema.payer_360_hcp_detail;

In [0]:
WITH qc_base AS (

SELECT *
FROM com_edp_prd.cmpa_insights_internal_schema.payer360_master

),

/* ======================================
QC 1 — National patient count
====================================== */
qc_national AS (

SELECT
'QC1_NATIONAL_PATIENTS' qc_check,
rollup_level,
SUM(total_elaprase_patients) metric_value
FROM qc_base
WHERE rollup_level='NATIONAL'
GROUP BY rollup_level
),

/* ======================================
QC 2 — Rollup totals
====================================== */
qc_rollup_totals AS (

SELECT
'QC2_ROLLUP_PATIENT_TOTALS' qc_check,
rollup_level,
SUM(total_elaprase_patients) metric_value
FROM qc_base
GROUP BY rollup_level
),

/* ======================================
QC 3 — Market share sums
====================================== */
qc_market_share AS (

SELECT
'QC3_MARKET_SHARE_SUM' qc_check,
rollup_level,
ROUND(SUM(payer_market_share_pct),2) metric_value
FROM qc_base
GROUP BY rollup_level
),

/* ======================================
QC 4 — Total lives check
====================================== */
qc_lives AS (

SELECT
'QC4_TOTAL_LIVES' qc_check,
rollup_level,
SUM(total_lives) metric_value
FROM qc_base
GROUP BY rollup_level
),

/* ======================================
QC 5 — Payer distribution
====================================== */
qc_top_payers AS (

SELECT
'QC5_TOP_PAYER_PATIENTS' qc_check,
payer_name,
SUM(total_elaprase_patients) metric_value
FROM qc_base
WHERE rollup_level='PAYER_ALL_TERRITORY'
GROUP BY payer_name
ORDER BY metric_value DESC
LIMIT 10
),

/* ======================================
QC 6 — Territory distribution
====================================== */
qc_top_territories AS (

SELECT
'QC6_TOP_TERRITORY_PATIENTS' qc_check,
territory_name,
SUM(total_elaprase_patients) metric_value
FROM qc_base
WHERE rollup_level='TERRITORY_ALL_PAYER'
GROUP BY territory_name
ORDER BY metric_value DESC
LIMIT 10
)

/* ======================================
FINAL QC OUTPUT
====================================== */
SELECT * FROM qc_national
UNION ALL
SELECT * FROM qc_rollup_totals
UNION ALL
SELECT * FROM qc_market_share
UNION ALL
SELECT * FROM qc_lives
UNION ALL
SELECT * FROM qc_top_payers
UNION ALL
SELECT * FROM qc_top_territories;

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.payer_360_hco_detail;

In [0]:
-- -- ============================================================
-- -- NATIONAL LEVEL MEDICARE PATIENTS – AGE VALIDATION
-- -- Keeping both MIN and MAX YOB
-- -- ============================================================

-- WITH demo_agg AS (
--     SELECT
--         PATIENT_ID,
--         PATIENT_GENDER,
--         MAX(PATIENT_YOB) AS MAX_PATIENT_YOB
--     FROM com_edp_prd.com_raw.kom_patient_demographics
--     GROUP BY PATIENT_ID, 2
-- )

-- SELECT
--     p.patient_id,

--     d.MAX_PATIENT_YOB,
--     PATIENT_GENDER,
--     /* Age using MAX YOB (younger age) */
--     FLOOR(DATEDIFF(CURRENT_DATE, d.MAX_PATIENT_YOB) / 365.25) AS age_using_max_yob

-- FROM com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level p

-- LEFT JOIN demo_agg d
--     ON p.patient_id = d.PATIENT_ID

-- WHERE UPPER(p.INSURANCE_GROUP) = 'MEDICARE';

In [0]:
-- -- ============================================================
-- -- QC 4: Rollup Reconciliation Check
-- -- ============================================================

-- /*
-- National rollup = Patient master count
-- Diff must be 0
-- */

-- WITH base AS (
--   SELECT COUNT(DISTINCT patient_id) AS base_patients
--   FROM payer_master_patient_level
-- ),

-- national AS (
--   SELECT total_elaprase_patients AS national_patients
--   FROM com_edp_prd.cmpa_insights_internal_schema.payer360_master
--   WHERE rollup_level = 'NATIONAL'
-- )

-- SELECT
--   b.base_patients,
--   n.national_patients,
--   (b.base_patients - n.national_patients) AS diff
-- FROM base b
-- CROSS JOIN national n;

### HCP & HCO Details Tables

In [0]:
CREATE OR REPLACE TEMP VIEW tx_enriched AS

SELECT
    a.*,

    CASE 
        WHEN t.code IN ('54092070001','540920700','J1743') THEN 'ELAPRASE'
        WHEN t.code = '8479600101' 
             OR t.code IN ('J3490','J3590','J9999') THEN 'AVLAYAH'
    END AS drug_type,

    CASE 
        WHEN UPPER(t.transaction_status) = 'REJECTED' THEN 1 
        ELSE 0 
    END AS is_denial

FROM all_patient_claims a   -- ✅ FIXED

LEFT JOIN (
    SELECT
        patient_id,
        medical_event_id AS claim_id,
        service_date AS fill_date,
        COALESCE(ndc11, procedure_code) AS code,
        'PAID' AS transaction_status
    FROM com_edp_prd.com_raw.kom_medical_events

    UNION ALL

    SELECT
        patient_id,
        pharmacy_event_id AS claim_id,
        fill_date,
        ndc11 AS code,
        transaction_result AS transaction_status
    FROM com_edp_prd.com_raw.kom_pharmacy_events
) t
ON a.patient_id = t.patient_id
AND a.claim_id = t.claim_id;

In [0]:
/*
PURPOSE
- Create HCO-level rollup metrics across Territory and Payer dimensions.
- Provide patient count, claims count, and most recent treatment date.
- Support multiple aggregation levels using GROUPING SETS.

BUSINESS LOGIC
1) Use enriched claim-level dataset (all_patient_claims_expanded).
2) Aggregate metrics at:
   - Territory + Payer + HCO (most granular; HCP removed)
   - Payer (all territories) with HCO breakdown
   - Territory (all payers)
   - National
3) Default rolled-up dimension values to 'ALL ...'.
4) Label each aggregation level using GROUPING().
*/

CREATE OR REPLACE TABLE cmpa_insights_internal_schema.PAYER_360_HCO_DETAIL AS

WITH patient_age AS (
SELECT
    patient_id,
    YEAR(CURRENT_DATE) - YEAR(patient_yob) AS patient_age
FROM (
    SELECT patient_id, MAX(patient_yob) AS patient_yob
    FROM com_edp_prd.com_raw.kom_patient_demographics
    GROUP BY patient_id
)
),

base AS (

SELECT
    a.*,

    pm.hco_veeva_crm_id,
    pm.hco_name,

    pm.territory_id,
    pm.territory AS territory_name,

    pa.patient_age,   -- ⚠️ required for LT17 logic

    pm.insurance_group,
    pm.payer_name AS payer_group,

    CAST(pm.payer_id AS STRING)  AS payer_id_rollup,
    CAST(pm.parent_id AS STRING) AS parent_id_rollup,
    pm.parent_name AS parent_name_rollup

FROM tx_enriched a

LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level pm
    ON a.patient_id = pm.patient_id

LEFT JOIN patient_age pa
    ON a.patient_id = pa.patient_id
),

/* ============================================================
1. TERRITORY + PAYER + HCO
============================================================ */

territory_payer AS (

SELECT

territory_id,
territory_name,
payer_id_rollup AS payer_id,
payer_group AS payer_name,

hco_veeva_crm_id,
MAX(hco_name) AS hco_name,

MAX(parent_id_rollup) AS parent_id,
MAX(parent_name_rollup) AS parent_name,

COUNT(DISTINCT patient_id) AS patient_count,

COUNT(DISTINCT CASE WHEN insurance_group='MEDICARE' THEN patient_id END) AS medicare_patients,
COUNT(DISTINCT CASE WHEN insurance_group='MEDICAID' THEN patient_id END) AS medicaid_patients,
COUNT(DISTINCT CASE WHEN insurance_group='COMMERCIAL' THEN patient_id END) AS commercial_patients,

COUNT(DISTINCT CASE WHEN patient_age < 17 THEN patient_id END) AS total_patients_lt17,

COUNT(DISTINCT CASE WHEN patient_age < 17 THEN claim_id END) AS total_claims_lt17,

COUNT(DISTINCT CASE
WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
OR insurance_group IS NULL
THEN patient_id
END) AS other_patients,

COUNT(DISTINCT claim_id) AS claims_count,
MAX(fill_date) AS last_treatment_date,

'TERRITORY_PAYER' AS rollup_level

FROM base

GROUP BY
territory_id,
territory_name,
payer_id_rollup,
payer_group,
hco_veeva_crm_id
),

/* ============================================================
2. PAYER ALL TERRITORY
============================================================ */

payer_all_territory AS (

SELECT

'ALL Territories' AS territory_id,
'All Territories' AS territory_name,

payer_id_rollup AS payer_id,
payer_group AS payer_name,

hco_veeva_crm_id,
MAX(hco_name) AS hco_name,

MAX(parent_id_rollup) AS parent_id,
MAX(parent_name_rollup) AS parent_name,

COUNT(DISTINCT patient_id) AS patient_count,

COUNT(DISTINCT CASE WHEN insurance_group='MEDICARE' THEN patient_id END) AS medicare_patients,
COUNT(DISTINCT CASE WHEN insurance_group='MEDICAID' THEN patient_id END) AS medicaid_patients,
COUNT(DISTINCT CASE WHEN insurance_group='COMMERCIAL' THEN patient_id END) AS commercial_patients,

COUNT(DISTINCT CASE WHEN patient_age < 17 THEN patient_id END) AS total_patients_lt17,

COUNT(DISTINCT CASE WHEN patient_age < 17 THEN claim_id END) AS total_claims_lt17,

COUNT(DISTINCT CASE
WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
OR insurance_group IS NULL
THEN patient_id
END) AS other_patients,

COUNT(DISTINCT claim_id) AS claims_count,
MAX(fill_date) AS last_treatment_date,

'PAYER_ALL_TERRITORY' AS rollup_level

FROM base

GROUP BY
payer_id_rollup,
payer_group,
hco_veeva_crm_id
),

/* ============================================================
3. TERRITORY ALL PAYER
============================================================ */

territory_all_payer AS (

SELECT

territory_id,
territory_name,

'ALL Payers' AS payer_id,
'All Payers' AS payer_name,

'ALL HCOs' AS hco_veeva_crm_id,
'All HCOs' AS hco_name,

'ALL Parents' AS parent_id,
'All Parents' AS parent_name,

COUNT(DISTINCT patient_id) AS patient_count,

COUNT(DISTINCT CASE WHEN insurance_group='MEDICARE' THEN patient_id END) AS medicare_patients,
COUNT(DISTINCT CASE WHEN insurance_group='MEDICAID' THEN patient_id END) AS medicaid_patients,
COUNT(DISTINCT CASE WHEN insurance_group='COMMERCIAL' THEN patient_id END) AS commercial_patients,

COUNT(DISTINCT CASE WHEN patient_age < 17 THEN patient_id END) AS total_patients_lt17,

COUNT(DISTINCT CASE WHEN patient_age < 17 THEN claim_id END) AS total_claims_lt17,

COUNT(DISTINCT CASE
WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
OR insurance_group IS NULL
THEN patient_id
END) AS other_patients,

COUNT(DISTINCT claim_id) AS claims_count,
MAX(fill_date) AS last_treatment_date,

'TERRITORY_ALL_PAYER' AS rollup_level

FROM base

GROUP BY
territory_id,
territory_name
),

/* ============================================================
4. NATIONAL
============================================================ */

national AS (

SELECT

'ALL Territories' AS territory_id,
'All Territories' AS territory_name,

'ALL Payers' AS payer_id,
'All Payers' AS payer_name,

'ALL HCOs' AS hco_veeva_crm_id,
'All HCOs' AS hco_name,

'ALL Parents' AS parent_id,
'All Parents' AS parent_name,

COUNT(DISTINCT patient_id) AS patient_count,

COUNT(DISTINCT CASE WHEN insurance_group='MEDICARE' THEN patient_id END) AS medicare_patients,
COUNT(DISTINCT CASE WHEN insurance_group='MEDICAID' THEN patient_id END) AS medicaid_patients,
COUNT(DISTINCT CASE WHEN insurance_group='COMMERCIAL' THEN patient_id END) AS commercial_patients,

COUNT(DISTINCT CASE WHEN patient_age < 17 THEN patient_id END) AS total_patients_lt17,

COUNT(DISTINCT CASE WHEN patient_age < 17 THEN claim_id END) AS total_claims_lt17,

COUNT(DISTINCT CASE
WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
OR insurance_group IS NULL
THEN patient_id
END) AS other_patients,

COUNT(DISTINCT claim_id) AS claims_count,
MAX(fill_date) AS last_treatment_date,

'NATIONAL' AS rollup_level

FROM base
),
ref_dedup AS (
    SELECT
        hco_veeva_crm_id,
        MAX(hco_city)  AS hco_city,
        MAX(hco_state) AS hco_state
    FROM com_edp_prd.cmpa_insights_internal_schema.reference_file
    GROUP BY hco_veeva_crm_id
)
/* ============================================================
FINAL OUTPUT
============================================================ */
SELECT 
    a.*,
    CASE 
        WHEN a.hco_veeva_crm_id = 'ALL HCOs' THEN 'ALL'
        ELSE ref.hco_city
    END AS hco_city,
    
    CASE 
        WHEN a.hco_veeva_crm_id = 'ALL HCOs' THEN 'ALL'
        ELSE ref.hco_state
    END AS hco_state

FROM (
    SELECT * FROM territory_payer
    UNION ALL
    SELECT * FROM payer_all_territory
    UNION ALL
    SELECT * FROM territory_all_payer
    UNION ALL
    SELECT * FROM national
) a

LEFT JOIN ref_dedup ref
    ON ref.hco_veeva_crm_id = a.hco_veeva_crm_id;

In [0]:
-- /*
-- PURPOSE
-- - Create a claim-level expanded dataset for eligible patients.
-- - Enrich each claim with HCP, HCO, payer, parent, territory, and region attributes.

-- BUSINESS LOGIC
-- 1) Start from all_patient_claims.
-- 2) Attach HCP + HCO info using reference file and provider table.
--    - If NPI is NULL → all HCP/HCO fields set to NULL.
--    - Prefer reference file values; fallback to provider table when needed.
-- 3) Map plan_id → payer and parent payer (default to 'Unknown' if missing).
-- 4) Map hcp_zip → territory and region (default to 'Unknown' if missing).
-- 5) Output one enriched row per patient claim.
-- */

-- CREATE OR REPLACE TEMPORARY VIEW all_patient_claims_expanded AS

-- /* ============================================================
--    1) Base claims
--    ============================================================ */
-- WITH t1 AS (
--   SELECT *
--   FROM all_patient_claims
-- ),

-- /* ============================================================
--    2) Attach HCP + HCO details
--    ============================================================ */
-- pulling_hco_affiliations AS (
--   SELECT
--     a.patient_id,
--     a.npi AS hcp_npi,
--     a.fill_date,
--     a.claim_id,
--     a.plan_id,

--     /* Derive HCP ZIP */
--     CASE
--       WHEN b.hcp_zip IS NULL OR b.hcp_zip = '-' THEN c.PROVIDER_ZIP
--       ELSE b.hcp_zip
--     END AS hcp_zip,

--     /* Derive HCP Name */
--     CASE
--       WHEN a.npi IS NULL THEN NULL
--       WHEN b.hcp_name IS NOT NULL THEN b.hcp_name
--       ELSE CONCAT(c.FIRST_NAME, ' ', c.LAST_NAME)
--     END AS hcp_name,

--     /* Specialty (NULL if no NPI) */
--     CASE 
--       WHEN a.npi IS NULL THEN NULL 
--       ELSE c.PRIMARY_SPECIALTY 
--     END AS hcp_specialty,

--     /* HCO fields (NULL if no NPI) */
--     CASE WHEN a.npi IS NULL THEN NULL ELSE b.hco_veeva_crm_id END AS hco_veeva_crm_id,
--     CASE WHEN a.npi IS NULL THEN NULL ELSE b.hco_name END AS hco_name

--   FROM t1 a
--   LEFT JOIN cmpa_insights_internal_schema.reference_file_0219 b
--     ON a.npi = b.hcp_npi
--   LEFT JOIN com_raw.kom_providers c
--     ON a.npi = c.npi 
--    AND c.PROVIDER_TYPE = 'INDIVIDUAL'
-- ),

-- /* ============================================================
--    3) Attach payer + parent payer attributes
--    ============================================================ */
-- payer_level_info AS (
--   SELECT
--     a.*,
--     COALESCE(b.PAYER_ID, 'Unknown')    AS payer_id,
--     COALESCE(b.PAYER_NAME, 'Unknown')  AS payer_name,
--     COALESCE(b.PARENT_ID, 'Unknown')   AS parent_id,
--     COALESCE(b.PARENT_NAME, 'Unknown') AS parent_name
--   FROM pulling_hco_affiliations a
--   LEFT JOIN com_raw.kom_plans b
--     ON a.plan_id = b.KH_PLAN_ID
-- ),

-- /* ============================================================
--    4) Attach territory + region from HCP ZIP
--    ============================================================ */
-- territory_level_info AS (
--   SELECT
--     a.*,
--     COALESCE(CAST(b.territory_id AS STRING), 'Unknown') AS territory_id,
--     COALESCE(b.territory_name, 'Unknown')                AS territory_name,
--     COALESCE(CAST(b.region_id AS STRING), 'Unknown')    AS region_id,
--     COALESCE(b.region_name, 'Unknown')                  AS region_name
--   FROM payer_level_info a
--   LEFT JOIN cmpa_insights_internal_schema.zip_to_territory_mapping b
--     ON a.hcp_zip = b.zipcode
-- )

-- /* ============================================================
--    Final Output: Claim-level enriched dataset
--    ============================================================ */
-- SELECT
--   patient_id,
--   claim_id,
--   fill_date,
--   hcp_npi,
--   hcp_name,
--   hcp_specialty,
--   hco_npi,
--   hco_name,
--   territory_id,
--   territory_name,
--   region_id,
--   region_name,
--   payer_id,
--   payer_name,
--   parent_id,
--   parent_name
-- FROM territory_level_info;

In [0]:
CREATE OR REPLACE TEMP VIEW all_patient_claims_expanded AS

SELECT
    c.patient_id,
    c.claim_id,
    c.fill_date,

    /* ================= HCP ================= */
    p.hcp_npi,
    p.hcp_name,
    p.hcp_specialty,

    /* ===============================
   STANDARDIZED HCO ATTRIBUTION
   (single canonical missing value)
   =============================== */

   CASE
      WHEN p.hco_veeva_crm_id IS NULL
         OR TRIM(p.hco_veeva_crm_id) = ''
         OR p.hco_veeva_crm_id = '-'
         OR UPPER(p.hco_veeva_crm_id) = 'UNKNOWN'
      THEN 'Unknown HCO'
      ELSE p.hco_veeva_crm_id
   END AS hco_veeva_crm_id,

   CASE
      WHEN p.hco_veeva_crm_id IS NULL
         OR TRIM(p.hco_veeva_crm_id) = ''
         OR p.hco_veeva_crm_id = '-'
         OR UPPER(p.hco_veeva_crm_id) = 'UNKNOWN'
      THEN 'Unknown HCO'
      ELSE COALESCE(p.hco_name,'Unknown HCO')
   END AS hco_name,

    /* ================= GEOGRAPHY ================= */
    p.territory_id,
    p.territory AS territory_name,
    p.region_id,
    p.region AS region_name,

    /* ================= PAYER GROUPING ================= */
   COALESCE(r.canonical_payer_id, CAST(p.PAYER_ID AS STRING)) AS PAYER_ID,
   COALESCE(r.canonical_payer_name, p.PAYER_NAME) AS PAYER_NAME,

   -- COALESCE(r.canonical_payer_id,'Unknown') AS parent_id,
   COALESCE(CAST(r.canonical_payer_id AS STRING),'Unknown') AS parent_id,
   COALESCE(r.canonical_payer_name,'Unknown') AS parent_name,
    (year(current_date)- year(pat_dem.patient_yob)) as patient_age,
    p.avlayah_pt_lt_17,
    p.elaprase_pt_lt_17

FROM all_patient_claims c

INNER JOIN com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level p
    ON c.patient_id = p.patient_id

LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.payer_rollup_dim r
   ON p.PAYER_ID = r.source_payer_id
LEFT JOIN (
        SELECT patient_id, MAX(patient_yob) patient_yob
        FROM com_edp_prd.com_raw.kom_patient_demographics
        GROUP BY patient_id
    ) pat_dem on c.patient_id = pat_dem.patient_id;

In [0]:
-- /*
-- PURPOSE
-- - Create HCP-level rollup metrics across Territory and Payer dimensions.
-- - Provide patient count, claims count, and most recent treatment date.
-- - Support multiple aggregation levels using GROUPING SETS.

-- BUSINESS LOGIC
-- 1) Use enriched claim-level dataset (all_patient_claims_expanded).
-- 2) Aggregate metrics at:
--    - Territory + Payer + HCP/HCO (most granular)
--    - Payer (all territories) with HCP/HCO breakdown
--    - Territory (all payers)
--    - National
-- 3) Default rolled-up dimension values to 'ALL ...'.
-- 4) Label each aggregation level using GROUPING().
-- */

-- CREATE OR REPLACE table cmpa_insights_internal_schema.PAYER_360_HCP_DETAIL AS

-- SELECT
--   COALESCE(territory_id, 'ALL Territories')  AS territory_id,
--   COALESCE(territory_name, 'All Territories') AS territory_name,
--   COALESCE(payer_id, 'ALL Payers')            AS payer_id,
--   COALESCE(payer_name, 'All Payers')          AS payer_name,
--   COALESCE(hcp_npi, 'ALL HCPs')               AS hcp_npi,
--   COALESCE(hcp_name, 'All HCPs')              AS hcp_name,
--   COALESCE(hcp_specialty, 'All HCPs')         AS hcp_specialty,
--   COALESCE(hco_npi, 'ALL HCOs')               AS hco_npi,
--   CASE 
--     WHEN GROUPING(hco_npi)=1 THEN 'All HCOs'
--     ELSE MAX(hco_name)
--   END AS hco_name,

--   /* Core Metrics */
--   COUNT(DISTINCT patient_id) AS patient_count,
--   COUNT(DISTINCT claim_id)   AS claims_count,
--   COUNT(DISTINCT hco_npi)    AS total_hcos, 
--   MAX(fill_date)             AS last_treatment_date,

--   /* Rollup Level Label */
--   CASE
--     WHEN GROUPING(territory_id) = 0 AND GROUPING(payer_id) = 0 THEN 'TERRITORY_PAYER'
--     WHEN GROUPING(territory_id) = 1 AND GROUPING(payer_id) = 0 THEN 'PAYER_ALL_TERRITORY'
--     WHEN GROUPING(territory_id) = 0 AND GROUPING(payer_id) = 1 THEN 'TERRITORY_ALL_PAYER'
--     WHEN GROUPING(territory_id) = 1 AND GROUPING(payer_id) = 1 THEN 'NATIONAL'
--   END AS rollup_level

-- FROM all_patient_claims_expanded

-- GROUP BY GROUPING SETS (

--   /* 1) Territory + Payer + HCP/HCO (most granular) */
--   (territory_id, territory_name, payer_id, payer_name, 
--    hcp_npi, hcp_name, hcp_specialty, hco_npi),

--   /* 2) Payer (all territories) with HCP/HCO breakdown */
--   (payer_id, payer_name, hcp_npi, hcp_name, hcp_specialty, hco_npi),

--   /* 3) Territory (all payers) */
--   (territory_id, territory_name),

--   /* 4) National */
--   ()
-- )

-- ORDER BY rollup_level, territory_name, payer_name, hcp_npi;

In [0]:
CREATE OR REPLACE TABLE cmpa_insights_internal_schema.PAYER_360_HCP_DETAIL AS

-- WITH base AS (

-- SELECT
--     a.*,

--     pm.insurance_group,
--     pm.payer_name AS payer_group,

--     CAST(pm.payer_id AS STRING)  AS payer_id_rollup,
--     CAST(pm.parent_id AS STRING) AS parent_id_rollup,
--     pm.parent_name AS parent_name_rollup

-- FROM tx_enriched a

-- LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level pm
--     ON a.patient_id = pm.patient_id
-- ),

WITH patient_age AS (
SELECT
    patient_id,
    YEAR(CURRENT_DATE) - YEAR(patient_yob) AS patient_age
FROM (
    SELECT patient_id, MAX(patient_yob) AS patient_yob
    FROM com_edp_prd.com_raw.kom_patient_demographics
    GROUP BY patient_id
)
),

base AS (

SELECT
    a.patient_id,
    a.claim_id,
    a.fill_date,
    pm.hcp_npi,

    /* bring HCP details if available in a */
    pm.hcp_name,
    pm.hcp_specialty,

    /* payer + territory + HCO */
    pm.territory_id,
    pm.territory AS territory_name,

    pm.hco_veeva_crm_id,
    pm.hco_name,

    pm.insurance_group,
    pm.payer_name AS payer_group,

    CAST(pm.payer_id AS STRING)  AS payer_id_rollup,
    CAST(pm.parent_id AS STRING) AS parent_id_rollup,
    pm.parent_name AS parent_name_rollup,

    /* age */
    pa.patient_age,

    /* bring flags if available in tx_enriched OR join from payer360 */
    pm.avlayah_pt_lt_17,
    pm.elaprase_pt_lt_17

FROM tx_enriched a

LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level pm
    ON a.patient_id = pm.patient_id

LEFT JOIN patient_age pa
    ON a.patient_id = pa.patient_id
),

/* ============================================================
1. TERRITORY + PAYER + HCP
============================================================ */

territory_payer AS (

SELECT

territory_id,
territory_name,
payer_id_rollup AS payer_id,
payer_group AS payer_name,

hcp_npi,
hcp_name,
hcp_specialty,

hco_veeva_crm_id,
MAX(hco_name) hco_name,

MAX(parent_id_rollup) parent_id,
MAX(parent_name_rollup) parent_name,

COUNT(DISTINCT patient_id) patient_count,

COUNT(DISTINCT CASE WHEN insurance_group='MEDICARE' THEN patient_id END) medicare_patients,
COUNT(DISTINCT CASE WHEN insurance_group='MEDICAID' THEN patient_id END) medicaid_patients,
COUNT(DISTINCT CASE WHEN insurance_group='COMMERCIAL' THEN patient_id END) commercial_patients,

COUNT(DISTINCT CASE WHEN patient_age < 17 THEN patient_id END) AS total_patients_lt17,

COUNT(DISTINCT CASE WHEN patient_age < 17 THEN claim_id END) AS total_claims_lt17,

COUNT(DISTINCT CASE
WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
OR insurance_group IS NULL
THEN patient_id
END) other_patients,

COUNT(DISTINCT CASE WHEN patient_age<5 THEN patient_id END) patient_count_age_less_than_5,
COUNT(DISTINCT CASE WHEN patient_age BETWEEN 5 AND 10 THEN patient_id END) patient_count_age_5_10,
COUNT(DISTINCT CASE WHEN patient_age BETWEEN 11 AND 16 THEN patient_id END) patient_count_age_11_16,
COUNT(DISTINCT CASE WHEN patient_age>=17 THEN patient_id END) patient_count_age_greater_than_17,

COUNT(DISTINCT CASE WHEN avlayah_pt_lt_17 = 1 THEN patient_id END) avlayah_pt_lt_17,
COUNT(DISTINCT CASE WHEN elaprase_pt_lt_17 = 1 THEN patient_id END) elaprase_pt_lt_17,

COUNT(DISTINCT claim_id) claims_count,
COUNT(DISTINCT hco_veeva_crm_id) total_hcos,
MAX(fill_date) last_treatment_date,

'TERRITORY_PAYER' rollup_level

FROM base

GROUP BY
territory_id,
territory_name,
payer_id_rollup,
payer_group,
hcp_npi,
hcp_name,
hcp_specialty,
hco_veeva_crm_id
),

/* ============================================================
2. PAYER ALL TERRITORY
============================================================ */

payer_all_territory AS (

SELECT

'ALL Territories' territory_id,
'All Territories' territory_name,

payer_id_rollup AS payer_id,
payer_group AS payer_name,

hcp_npi,
hcp_name,
hcp_specialty,

hco_veeva_crm_id,
MAX(hco_name) hco_name,

MAX(parent_id_rollup) parent_id,
MAX(parent_name_rollup) parent_name,

COUNT(DISTINCT patient_id) patient_count,

COUNT(DISTINCT CASE WHEN insurance_group='MEDICARE' THEN patient_id END) medicare_patients,
COUNT(DISTINCT CASE WHEN insurance_group='MEDICAID' THEN patient_id END) medicaid_patients,
COUNT(DISTINCT CASE WHEN insurance_group='COMMERCIAL' THEN patient_id END) commercial_patients,

COUNT(DISTINCT CASE WHEN patient_age < 17 THEN patient_id END) AS total_patients_lt17,

COUNT(DISTINCT CASE WHEN patient_age < 17 THEN claim_id END) AS total_claims_lt17,

COUNT(DISTINCT CASE
WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
OR insurance_group IS NULL
THEN patient_id
END) other_patients,

COUNT(DISTINCT CASE WHEN patient_age<5 THEN patient_id END) patient_count_age_less_than_5,
COUNT(DISTINCT CASE WHEN patient_age BETWEEN 5 AND 10 THEN patient_id END) patient_count_age_5_10,
COUNT(DISTINCT CASE WHEN patient_age BETWEEN 11 AND 16 THEN patient_id END) patient_count_age_11_16,
COUNT(DISTINCT CASE WHEN patient_age>=17 THEN patient_id END) patient_count_age_greater_than_17,

COUNT(DISTINCT CASE WHEN avlayah_pt_lt_17 = 1 THEN patient_id END) avlayah_pt_lt_17,
COUNT(DISTINCT CASE WHEN elaprase_pt_lt_17 = 1 THEN patient_id END) elaprase_pt_lt_17,

COUNT(DISTINCT claim_id) claims_count,
COUNT(DISTINCT hco_veeva_crm_id) total_hcos,
MAX(fill_date) last_treatment_date,

'PAYER_ALL_TERRITORY' rollup_level

FROM base

GROUP BY
payer_id_rollup,
payer_group,
hcp_npi,
hcp_name,
hcp_specialty,
hco_veeva_crm_id
),

/* ============================================================
3. TERRITORY ALL PAYER
============================================================ */

territory_all_payer AS (

SELECT

territory_id,
territory_name,

'ALL Payers' payer_id,
'All Payers' payer_name,

'ALL HCPs' hcp_npi,
'All HCPs' hcp_name,
'All HCPs' hcp_specialty,

'ALL HCOs' hco_veeva_crm_id,
'All HCOs' hco_name,

'ALL Parents' parent_id,
'All Parents' parent_name,

COUNT(DISTINCT patient_id) patient_count,

COUNT(DISTINCT CASE WHEN insurance_group='MEDICARE' THEN patient_id END) medicare_patients,
COUNT(DISTINCT CASE WHEN insurance_group='MEDICAID' THEN patient_id END) medicaid_patients,
COUNT(DISTINCT CASE WHEN insurance_group='COMMERCIAL' THEN patient_id END) commercial_patients,

COUNT(DISTINCT CASE WHEN patient_age < 17 THEN patient_id END) AS total_patients_lt17,

COUNT(DISTINCT CASE WHEN patient_age < 17 THEN claim_id END) AS total_claims_lt17,

COUNT(DISTINCT CASE
WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
OR insurance_group IS NULL
THEN patient_id
END) other_patients,

COUNT(DISTINCT CASE WHEN patient_age<5 THEN patient_id END) patient_count_age_less_than_5,
COUNT(DISTINCT CASE WHEN patient_age BETWEEN 5 AND 10 THEN patient_id END) patient_count_age_5_10,
COUNT(DISTINCT CASE WHEN patient_age BETWEEN 11 AND 16 THEN patient_id END) patient_count_age_11_16,
COUNT(DISTINCT CASE WHEN patient_age>=17 THEN patient_id END) patient_count_age_greater_than_17,

COUNT(DISTINCT CASE WHEN avlayah_pt_lt_17 = 1 THEN patient_id END) avlayah_pt_lt_17,
COUNT(DISTINCT CASE WHEN elaprase_pt_lt_17 = 1 THEN patient_id END) elaprase_pt_lt_17,

COUNT(DISTINCT claim_id) claims_count,
COUNT(DISTINCT hco_veeva_crm_id) total_hcos,
MAX(fill_date) last_treatment_date,

'TERRITORY_ALL_PAYER' rollup_level

FROM base

GROUP BY
territory_id,
territory_name
),

/* ============================================================
4. NATIONAL
============================================================ */

national AS (

SELECT

'ALL Territories' territory_id,
'All Territories' territory_name,

'ALL Payers' payer_id,
'All Payers' payer_name,

'ALL HCPs' hcp_npi,
'All HCPs' hcp_name,
'All HCPs' hcp_specialty,

'ALL HCOs' hco_veeva_crm_id,
'All HCOs' hco_name,

'ALL Parents' parent_id,
'All Parents' parent_name,

COUNT(DISTINCT patient_id) patient_count,

COUNT(DISTINCT CASE WHEN insurance_group='MEDICARE' THEN patient_id END) medicare_patients,
COUNT(DISTINCT CASE WHEN insurance_group='MEDICAID' THEN patient_id END) medicaid_patients,
COUNT(DISTINCT CASE WHEN insurance_group='COMMERCIAL' THEN patient_id END) commercial_patients,

COUNT(DISTINCT CASE WHEN patient_age < 17 THEN patient_id END) AS total_patients_lt17,

COUNT(DISTINCT CASE WHEN patient_age < 17 THEN claim_id END) AS total_claims_lt17,

COUNT(DISTINCT CASE
WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
OR insurance_group IS NULL
THEN patient_id
END) other_patients,

COUNT(DISTINCT CASE WHEN patient_age<5 THEN patient_id END) patient_count_age_less_than_5,
COUNT(DISTINCT CASE WHEN patient_age BETWEEN 5 AND 10 THEN patient_id END) patient_count_age_5_10,
COUNT(DISTINCT CASE WHEN patient_age BETWEEN 11 AND 16 THEN patient_id END) patient_count_age_11_16,
COUNT(DISTINCT CASE WHEN patient_age>=17 THEN patient_id END) patient_count_age_greater_than_17,

COUNT(DISTINCT CASE WHEN avlayah_pt_lt_17 = 1 THEN patient_id END) avlayah_pt_lt_17,
COUNT(DISTINCT CASE WHEN elaprase_pt_lt_17 = 1 THEN patient_id END) elaprase_pt_lt_17,

COUNT(DISTINCT claim_id) claims_count,
COUNT(DISTINCT hco_veeva_crm_id) total_hcos,
MAX(fill_date) last_treatment_date,

'NATIONAL' rollup_level

FROM base
)

SELECT * FROM territory_payer
UNION ALL
SELECT * FROM payer_all_territory
UNION ALL
SELECT * FROM territory_all_payer
UNION ALL
SELECT * FROM national;

In [0]:
select * from cmpa_insights_internal_schema.PAYER_360_HCP_DETAIL

In [0]:
CREATE OR REPLACE table cmpa_insights_internal_schema.PAYER_360_HCP_DETAIL_PATIENT_LEVEL AS

WITH patient_age_map AS (
    SELECT 
        patient_id,
        YEAR(CURRENT_DATE) - YEAR(MAX(patient_yob)) AS patient_age
    FROM com_edp_prd.com_raw.kom_patient_demographics
    GROUP BY patient_id
),

base AS (
    SELECT
        a.*,

        /* ✅ territory + provider */
        pm.territory_id,
        pm.territory AS territory_name,

        pm.hcp_npi,
        pm.hcp_name,
        pm.hcp_specialty,

        pm.hco_veeva_crm_id,
        pm.hco_name,

        /* payer */
        pm.insurance_group,
        pm.payer_name AS payer_group,

        CAST(pm.payer_id  AS STRING) AS payer_id_rollup,
        CAST(pm.parent_id AS STRING) AS parent_id_rollup,
        pm.parent_name             AS parent_name_rollup,

        /* ✅ FIXED: age comes from demographics, NOT pm */
        pa.patient_age

    FROM tx_enriched a

    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level pm
        ON a.patient_id = pm.patient_id

    LEFT JOIN patient_age_map pa
        ON a.patient_id = pa.patient_id
),

/* ============================================================
   1. TERRITORY + PAYER  (patient-level)
   Join key: territory_id, payer_id, hcp_npi, hco_npi
============================================================ */
territory_payer AS (
    SELECT
        territory_id,
        territory_name,
        payer_id_rollup   AS payer_id,
        payer_group       AS payer_name,
        hcp_npi,
        hcp_name,
        hcp_specialty,
        hco_veeva_crm_id,
        hco_name,
        parent_id_rollup  AS parent_id,
        parent_name_rollup AS parent_name,
        patient_id,
        insurance_group,
        claim_id,
        fill_date,
        patient_age,
        'TERRITORY_PAYER'  AS rollup_level
    FROM base
),

/* ============================================================
   2. PAYER ALL TERRITORY  (patient-level)
   Territory collapsed → 'ALL Territories'
   Join key: payer_id, hcp_npi, hco_npi
============================================================ */
payer_all_territory AS (
    SELECT
        'ALL Territories'  AS territory_id,
        'All Territories'  AS territory_name,
        payer_id_rollup    AS payer_id,
        payer_group        AS payer_name,
        hcp_npi,
        hcp_name,
        hcp_specialty,
        hco_veeva_crm_id,
        hco_name,
        parent_id_rollup   AS parent_id,
        parent_name_rollup AS parent_name,
        patient_id,
        insurance_group,
        claim_id,
        fill_date,
        patient_age,
        'PAYER_ALL_TERRITORY' AS rollup_level
    FROM base
),

/* ============================================================
   3. TERRITORY ALL PAYER  (patient-level)
   Payer/HCP/HCO/Parent collapsed → 'ALL ...'
   Join key: territory_id
============================================================ */
territory_all_payer AS (
    SELECT
        territory_id,
        territory_name,
        'ALL Payers'   AS payer_id,
        'All Payers'   AS payer_name,
        'ALL HCPs'     AS hcp_npi,
        'All HCPs'     AS hcp_name,
        'All HCPs'     AS hcp_specialty,
        'ALL HCOs'     AS hco_veeva_crm_id,
        'All HCOs'     AS hco_name,
        'ALL Parents'  AS parent_id,
        'All Parents'  AS parent_name,
        patient_id,
        insurance_group,
        claim_id,
        fill_date,
        patient_age,
        'TERRITORY_ALL_PAYER' AS rollup_level
    FROM base
),

/* ============================================================
   4. NATIONAL  (patient-level)
   Everything collapsed → 'ALL ...'
   Join key: rollup_level only
============================================================ */
national AS (
    SELECT
        'ALL Territories' AS territory_id,
        'All Territories' AS territory_name,
        'ALL Payers'      AS payer_id,
        'All Payers'      AS payer_name,
        'ALL HCPs'        AS hcp_npi,
        'All HCPs'        AS hcp_name,
        'All HCPs'        AS hcp_specialty,
        'ALL HCOs'        AS hco_veeva_crm_id,
        'All HCOs'        AS hco_name,
        'ALL Parents'     AS parent_id,
        'All Parents'     AS parent_name,
        patient_id,
        insurance_group,
        claim_id,
        fill_date,
        patient_age,
        'NATIONAL' AS rollup_level
    FROM base
),

final_rollups as (
  SELECT * FROM territory_payer
UNION ALL
SELECT * FROM payer_all_territory
UNION ALL
SELECT * FROM territory_all_payer
UNION ALL
SELECT * FROM national
)

SELECT DISTINCT
    territory_id,
    territory_name,
    payer_id,
    payer_name,
    hcp_npi,
    hcp_name,
    hcp_specialty,
    hco_veeva_crm_id,
    hco_name,
    parent_id,
    parent_name,
    patient_id,
    insurance_group,
    fill_date,
    patient_age,
    rollup_level
FROM final_rollups

In [0]:
/*
PURPOSE
- Create HCO-level rollup metrics across Territory and Payer dimensions.
- Provide patient count, claims count, and most recent treatment date.
- Support multiple aggregation levels using GROUPING SETS.

BUSINESS LOGIC
1) Use enriched claim-level dataset (tx_enriched).
2) Aggregate metrics at:
   - Territory + Payer + HCO (most granular; HCP removed)
   - Payer (all territories) with HCO breakdown
   - Territory (all payers)
   - National
3) Default rolled-up dimension values to 'ALL ...'.
4) Label each aggregation level using GROUPING().
*/

CREATE OR REPLACE TABLE cmpa_insights_internal_schema.PAYER_360_HCO_DETAIL AS

WITH patient_age AS (
    SELECT
        patient_id,
        YEAR(CURRENT_DATE) - YEAR(patient_yob) AS patient_age
    FROM (
        SELECT 
            patient_id, 
            MAX(patient_yob) AS patient_yob
        FROM com_edp_prd.com_raw.kom_patient_demographics
        GROUP BY patient_id
    )
),

base AS (
    SELECT
        a.*,

        /* ✅ ADD THESE */
        pm.territory_id,
        pm.territory AS territory_name,

        pm.hcp_npi,
        pm.hcp_name,
        pm.hcp_specialty,

        pm.hco_veeva_crm_id,
        pm.hco_name,

        pm.insurance_group,
        pm.payer_name AS payer_group,

        pa.patient_age,

        CAST(pm.payer_id   AS STRING) AS payer_id_rollup,
        CAST(pm.parent_id  AS STRING) AS parent_id_rollup,
        pm.parent_name                AS parent_name_rollup

    FROM tx_enriched a

    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level pm
        ON a.patient_id = pm.patient_id

    LEFT JOIN patient_age pa
        ON a.patient_id = pa.patient_id
),

/* ============================================================
1. TERRITORY + PAYER + HCO
============================================================ */

territory_payer AS (

SELECT

territory_id,
territory_name,
payer_id_rollup AS payer_id,
payer_group AS payer_name,

hco_veeva_crm_id,
MAX(hco_name) AS hco_name,

MAX(parent_id_rollup) AS parent_id,
MAX(parent_name_rollup) AS parent_name,

COUNT(DISTINCT patient_id) AS patient_count,

COUNT(DISTINCT CASE WHEN insurance_group='MEDICARE' THEN patient_id END) AS medicare_patients,
COUNT(DISTINCT CASE WHEN insurance_group='MEDICAID' THEN patient_id END) AS medicaid_patients,
COUNT(DISTINCT CASE WHEN insurance_group='COMMERCIAL' THEN patient_id END) AS commercial_patients,

COUNT(DISTINCT CASE WHEN patient_age < 17 THEN patient_id END) AS total_patients_lt17,

COUNT(DISTINCT CASE WHEN patient_age < 17 THEN claim_id END) AS total_claims_lt17,

COUNT(DISTINCT CASE
WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
OR insurance_group IS NULL
THEN patient_id
END) AS other_patients,

COUNT(DISTINCT claim_id) AS claims_count,
MAX(fill_date) AS last_treatment_date,

'TERRITORY_PAYER' AS rollup_level

FROM base

GROUP BY
territory_id,
territory_name,
payer_id_rollup,
payer_group,
hco_veeva_crm_id
),

/* ============================================================
2. PAYER ALL TERRITORY
============================================================ */

payer_all_territory AS (

SELECT

'ALL Territories' AS territory_id,
'All Territories' AS territory_name,

payer_id_rollup AS payer_id,
payer_group AS payer_name,

hco_veeva_crm_id,
MAX(hco_name) AS hco_name,

MAX(parent_id_rollup) AS parent_id,
MAX(parent_name_rollup) AS parent_name,

COUNT(DISTINCT patient_id) AS patient_count,

COUNT(DISTINCT CASE WHEN insurance_group='MEDICARE' THEN patient_id END) AS medicare_patients,
COUNT(DISTINCT CASE WHEN insurance_group='MEDICAID' THEN patient_id END) AS medicaid_patients,
COUNT(DISTINCT CASE WHEN insurance_group='COMMERCIAL' THEN patient_id END) AS commercial_patients,

COUNT(DISTINCT CASE WHEN patient_age < 17 THEN patient_id END) AS total_patients_lt17,

COUNT(DISTINCT CASE WHEN patient_age < 17 THEN claim_id END) AS total_claims_lt17,

COUNT(DISTINCT CASE
WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
OR insurance_group IS NULL
THEN patient_id
END) AS other_patients,

COUNT(DISTINCT claim_id) AS claims_count,
MAX(fill_date) AS last_treatment_date,

'PAYER_ALL_TERRITORY' AS rollup_level

FROM base

GROUP BY
payer_id_rollup,
payer_group,
hco_veeva_crm_id
),

/* ============================================================
3. TERRITORY ALL PAYER
============================================================ */

territory_all_payer AS (

SELECT

territory_id,
territory_name,

'ALL Payers' AS payer_id,
'All Payers' AS payer_name,

'ALL HCOs' AS hco_veeva_crm_id,
'All HCOs' AS hco_name,

'ALL Parents' AS parent_id,
'All Parents' AS parent_name,

COUNT(DISTINCT patient_id) AS patient_count,

COUNT(DISTINCT CASE WHEN insurance_group='MEDICARE' THEN patient_id END) AS medicare_patients,
COUNT(DISTINCT CASE WHEN insurance_group='MEDICAID' THEN patient_id END) AS medicaid_patients,
COUNT(DISTINCT CASE WHEN insurance_group='COMMERCIAL' THEN patient_id END) AS commercial_patients,

COUNT(DISTINCT CASE WHEN patient_age < 17 THEN patient_id END) AS total_patients_lt17,

COUNT(DISTINCT CASE WHEN patient_age < 17 THEN claim_id END) AS total_claims_lt17,

COUNT(DISTINCT CASE
WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
OR insurance_group IS NULL
THEN patient_id
END) AS other_patients,

COUNT(DISTINCT claim_id) AS claims_count,
MAX(fill_date) AS last_treatment_date,

'TERRITORY_ALL_PAYER' AS rollup_level

FROM base

GROUP BY
territory_id,
territory_name
),

/* ============================================================
4. NATIONAL
============================================================ */

national AS (

SELECT

'ALL Territories' AS territory_id,
'All Territories' AS territory_name,

'ALL Payers' AS payer_id,
'All Payers' AS payer_name,

'ALL HCOs' AS hco_veeva_crm_id,
'All HCOs' AS hco_name,

'ALL Parents' AS parent_id,
'All Parents' AS parent_name,

COUNT(DISTINCT patient_id) AS patient_count,

COUNT(DISTINCT CASE WHEN insurance_group='MEDICARE' THEN patient_id END) AS medicare_patients,
COUNT(DISTINCT CASE WHEN insurance_group='MEDICAID' THEN patient_id END) AS medicaid_patients,
COUNT(DISTINCT CASE WHEN insurance_group='COMMERCIAL' THEN patient_id END) AS commercial_patients,

COUNT(DISTINCT CASE WHEN patient_age < 17 THEN patient_id END) AS total_patients_lt17,

COUNT(DISTINCT CASE WHEN patient_age < 17 THEN claim_id END) AS total_claims_lt17,

COUNT(DISTINCT CASE
WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
OR insurance_group IS NULL
THEN patient_id
END) AS other_patients,

COUNT(DISTINCT claim_id) AS claims_count,
MAX(fill_date) AS last_treatment_date,

'NATIONAL' AS rollup_level

FROM base
),
ref_dedup AS (
    SELECT
        hco_veeva_crm_id,
        MAX(hco_city)  AS hco_city,
        MAX(hco_state) AS hco_state
    FROM com_edp_prd.cmpa_insights_internal_schema.reference_file
    GROUP BY hco_veeva_crm_id
)
/* ============================================================
FINAL OUTPUT
============================================================ */
SELECT 
    a.*,
    CASE 
        WHEN a.hco_veeva_crm_id = 'ALL HCOs' THEN 'ALL'
        ELSE ref.hco_city
    END AS hco_city,
    
    CASE 
        WHEN a.hco_veeva_crm_id = 'ALL HCOs' THEN 'ALL'
        ELSE ref.hco_state
    END AS hco_state

FROM (
    SELECT * FROM territory_payer
    UNION ALL
    SELECT * FROM payer_all_territory
    UNION ALL
    SELECT * FROM territory_all_payer
    UNION ALL
    SELECT * FROM national
) a

LEFT JOIN ref_dedup ref
    ON ref.hco_veeva_crm_id = a.hco_veeva_crm_id;

In [0]:
select * from cmpa_insights_internal_schema.PAYER_360_HCO_DETAIL ;

In [0]:
WITH tx_classification AS (

SELECT
    patient_id,
    claim_id,
    fill_date,

    /* ================= DRUG FLAG ================= */
    CASE 
        WHEN 
            /* ELAPRASE */
            (
                code IN ('54092070001','540920700','J1743')
            )
        THEN 'ELAPRASE'

        WHEN 
            /* AVLayah */
            (
                code = '8479600101'
                OR code IN ('J3490','J3590','J9999')
            )
        THEN 'AVLAYAH'

        ELSE NULL
    END AS drug_type,

    /* ================= DENIAL FLAG ================= */
    CASE 
        WHEN UPPER(transaction_status) = 'REJECTED' THEN 1
        ELSE 0
    END AS is_denial

FROM (

    /* MEDICAL */
    SELECT
        patient_id,
        medical_event_id AS claim_id,
        service_date AS fill_date,
        COALESCE(ndc11, procedure_code) AS code,
        'MEDICAL' AS claim_source,
        'PAID' AS transaction_status
    FROM com_edp_prd.com_raw.kom_medical_events

    UNION ALL

    /* PHARMACY */
    SELECT
        patient_id,
        pharmacy_event_id AS claim_id,
        fill_date,
        ndc11 AS code,
        'PHARMACY',
        transaction_result AS transaction_status
    FROM com_edp_prd.com_raw.kom_pharmacy_events

)
WHERE code IS NOT NULL
),

patient_age_map AS (
    SELECT 
        patient_id,
        YEAR(CURRENT_DATE) - YEAR(MAX(patient_yob)) AS patient_age
    FROM com_edp_prd.com_raw.kom_patient_demographics
    GROUP BY patient_id
),

-- patient_age_map AS (
--     SELECT 
--         patient_id,
--         CASE 
--             WHEN MAX(patient_yob) IS NOT NULL 
--             THEN CAST(date_format(CURRENT_DATE, 'yyyy') AS INT) - CAST(MAX(patient_yob) AS INT)
--         END AS patient_age
--     FROM com_edp_prd.com_raw.kom_patient_demographics
--     GROUP BY patient_id
-- ),
-- patient_age_map AS (
--     SELECT 
--         patient_id,
--         YEAR(CURRENT_DATE) - MAX(patient_yob) AS patient_age
--     FROM com_edp_prd.com_raw.kom_patient_demographics
--     GROUP BY patient_id
-- ),

patient_tx_enriched AS (
SELECT 
    p.*,
    a.patient_age,
    t.claim_id,
    t.fill_date,
    t.drug_type,
    t.is_denial

FROM com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level p

LEFT JOIN tx_classification t
    ON p.patient_id = t.patient_id

LEFT JOIN patient_age_map a
    ON p.patient_id = a.patient_id
),

 patient_tx_metrics AS (

SELECT

    /* ================= BASE ================= */
    COUNT(DISTINCT CASE WHEN patient_age < 17 THEN patient_id END) AS total_patients_lt17,

    COUNT(DISTINCT CASE 
        WHEN patient_age < 17 THEN claim_id END) AS total_claims_lt17,

    /* ================= ELAPRASE ================= */
    COUNT(DISTINCT CASE 
        WHEN patient_age < 17 AND drug_type = 'ELAPRASE'
        THEN patient_id END) AS elaprase_pt_ct_lt17,

    COUNT(DISTINCT CASE 
        WHEN patient_age < 17 AND drug_type = 'ELAPRASE'
        THEN claim_id END) AS elaprase_claims_ct,

    COUNT(DISTINCT CASE 
        WHEN drug_type = 'ELAPRASE' AND insurance_group = 'COMMERCIAL'
        THEN patient_id END) AS elaprase_commercial_pt_ct,

    COUNT(DISTINCT CASE 
        WHEN drug_type = 'ELAPRASE' AND insurance_group = 'MEDICARE'
        THEN patient_id END) AS elaprase_medicare_pt_ct,

    COUNT(DISTINCT CASE 
        WHEN drug_type = 'ELAPRASE' 
        AND insurance_group NOT IN ('MEDICARE','COMMERCIAL')
        THEN patient_id END) AS elaprase_other_pt_ct,

    /* ================= AVLayah ================= */
    COUNT(DISTINCT CASE 
        WHEN patient_age < 17 AND drug_type = 'AVLAYAH'
        THEN patient_id END) AS avlayah_pt_ct_lt17,

    COUNT(DISTINCT CASE 
        WHEN drug_type = 'AVLAYAH'
        THEN claim_id END) AS avlayah_claims_ct,

    COUNT(DISTINCT CASE 
        WHEN drug_type = 'AVLAYAH' AND insurance_group = 'COMMERCIAL'
        THEN patient_id END) AS avlayah_commercial_pt_ct,

    COUNT(DISTINCT CASE 
        WHEN drug_type = 'AVLAYAH' AND insurance_group = 'MEDICARE'
        THEN patient_id END) AS avlayah_medicare_pt_ct,

    COUNT(DISTINCT CASE 
        WHEN drug_type = 'AVLAYAH' 
        AND insurance_group NOT IN ('MEDICARE','COMMERCIAL')
        THEN patient_id END) AS avlayah_other_pt_ct,

    /* ================= DENIAL RATES ================= */
    CASE 
        WHEN COUNT(CASE WHEN drug_type='ELAPRASE' THEN claim_id END) = 0 THEN 0
        ELSE ROUND(
            100.0 * SUM(CASE WHEN drug_type='ELAPRASE' AND is_denial=1 THEN 1 ELSE 0 END)
            / COUNT(CASE WHEN drug_type='ELAPRASE' THEN claim_id END), 2)
    END AS elaprase_denial_rate,

    CASE 
        WHEN COUNT(CASE WHEN drug_type='AVLAYAH' THEN claim_id END) = 0 THEN 0
        ELSE ROUND(
            100.0 * SUM(CASE WHEN drug_type='AVLAYAH' AND is_denial=1 THEN 1 ELSE 0 END)
            / COUNT(CASE WHEN drug_type='AVLAYAH' THEN claim_id END), 2)
    END AS avlayah_denial_rate,

    /* ================= HCP / HCO ================= */
    COUNT(DISTINCT CASE WHEN drug_type='ELAPRASE' THEN hcp_npi END) AS elaprase_hcp,
    COUNT(DISTINCT CASE WHEN drug_type='AVLAYAH' THEN hcp_npi END) AS avlayah_hcp,

    COUNT(DISTINCT CASE WHEN drug_type='ELAPRASE' THEN hco_veeva_crm_id END) AS elaprase_hco,
    COUNT(DISTINCT CASE WHEN drug_type='AVLAYAH' THEN hco_veeva_crm_id END) AS avlayah_hco

FROM patient_tx_enriched
)
SELECT * FROM patient_tx_metrics;

In [0]:
CREATE OR REPLACE TABLE cmpa_insights_internal_schema.patient360_master_enriched AS

WITH tx_classification AS (
    SELECT
        patient_id,
        claim_id,
        fill_date,

        CASE 
            WHEN code IN ('54092070001','540920700','J1743') THEN 'ELAPRASE'
            WHEN code = '8479600101' 
                 OR code IN ('J3490','J3590','J9999') THEN 'AVLAYAH'
        END AS drug_type,

        CASE 
            WHEN UPPER(transaction_status) = 'REJECTED' THEN 1 
            ELSE 0 
        END AS is_denial

    FROM (
        SELECT
            patient_id,
            medical_event_id AS claim_id,
            service_date AS fill_date,
            COALESCE(ndc11, procedure_code) AS code,
            'PAID' AS transaction_status
        FROM com_edp_prd.com_raw.kom_medical_events

        UNION ALL

        SELECT
            patient_id,
            pharmacy_event_id AS claim_id,
            fill_date,
            ndc11 AS code,
            transaction_result AS transaction_status
        FROM com_edp_prd.com_raw.kom_pharmacy_events
    )
),

patient_age_map AS (
    SELECT 
        patient_id,
        YEAR(CURRENT_DATE()) - YEAR(MAX(patient_yob)) AS patient_age
    FROM com_edp_prd.com_raw.kom_patient_demographics
    GROUP BY patient_id
),

base AS (
    SELECT 
        p.patient_id,
        p.insurance_group,
        a.patient_age,
        t.claim_id,
        t.drug_type,
        t.is_denial
    FROM com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level p
    LEFT JOIN tx_classification t 
        ON p.patient_id = t.patient_id
    LEFT JOIN patient_age_map a 
        ON p.patient_id = a.patient_id
),

patient_tx_patient_level AS (
    SELECT
        patient_id,

        /* BASE */
        COUNT(DISTINCT CASE WHEN patient_age < 17 THEN claim_id END) AS total_claims_lt17,
        CASE WHEN MAX(patient_age) < 17 THEN 1 ELSE 0 END AS total_patients_lt17,

        /* ELAPRASE */
        MAX(CASE WHEN patient_age < 17 AND drug_type='ELAPRASE' THEN 1 ELSE 0 END) AS elaprase_pt_ct_lt17,
        COUNT(DISTINCT CASE WHEN drug_type='ELAPRASE' THEN claim_id END) AS elaprase_claims_ct,

        /* AVLAYAH */
        MAX(CASE WHEN patient_age < 17 AND drug_type='AVLAYAH' THEN 1 ELSE 0 END) AS avlayah_pt_ct_lt17,
        COUNT(DISTINCT CASE WHEN drug_type='AVLAYAH' THEN claim_id END) AS avlayah_claims_ct,

        /* INSURANCE SPLIT */
        MAX(CASE WHEN drug_type='ELAPRASE' AND insurance_group='COMMERCIAL' THEN 1 ELSE 0 END) AS elaprase_commercial_pt_ct,
        MAX(CASE WHEN drug_type='ELAPRASE' AND insurance_group='MEDICARE' THEN 1 ELSE 0 END) AS elaprase_medicare_pt_ct,
        MAX(CASE WHEN drug_type='ELAPRASE' AND insurance_group NOT IN ('MEDICARE','COMMERCIAL') THEN 1 ELSE 0 END) AS elaprase_other_pt_ct,

        MAX(CASE WHEN drug_type='AVLAYAH' AND insurance_group='COMMERCIAL' THEN 1 ELSE 0 END) AS avlayah_commercial_pt_ct,
        MAX(CASE WHEN drug_type='AVLAYAH' AND insurance_group='MEDICARE' THEN 1 ELSE 0 END) AS avlayah_medicare_pt_ct,
        MAX(CASE WHEN drug_type='AVLAYAH' AND insurance_group NOT IN ('MEDICARE','COMMERCIAL') THEN 1 ELSE 0 END) AS avlayah_other_pt_ct,

        /* DENIAL RATE */
        CASE 
            WHEN COUNT(CASE WHEN drug_type='ELAPRASE' THEN claim_id END)=0 THEN 0
            ELSE ROUND(
                100.0 * SUM(CASE WHEN drug_type='ELAPRASE' AND is_denial=1 THEN 1 ELSE 0 END)
                / COUNT(CASE WHEN drug_type='ELAPRASE' THEN claim_id END),2)
        END AS elaprase_denial_rate,

        CASE 
            WHEN COUNT(CASE WHEN drug_type='AVLAYAH' THEN claim_id END)=0 THEN 0
            ELSE ROUND(
                100.0 * SUM(CASE WHEN drug_type='AVLAYAH' AND is_denial=1 THEN 1 ELSE 0 END)
                / COUNT(CASE WHEN drug_type='AVLAYAH' THEN claim_id END),2)
        END AS avlayah_denial_rate

    FROM base
    GROUP BY patient_id
)

SELECT
    m.*,

    t.elaprase_pt_ct_lt17,
    t.avlayah_pt_ct_lt17,
    t.elaprase_denial_rate,
    t.avlayah_denial_rate,

    t.avlayah_commercial_pt_ct,
    t.avlayah_medicare_pt_ct,
    t.avlayah_other_pt_ct,

    t.elaprase_commercial_pt_ct,
    t.elaprase_medicare_pt_ct,
    t.elaprase_other_pt_ct,

    t.total_claims_lt17,
    t.total_patients_lt17,

    t.avlayah_claims_ct,
    t.elaprase_claims_ct

FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master m
LEFT JOIN patient_tx_patient_level t
    ON m.patient_id = t.patient_id;

In [0]:
with patient_age_map AS (
    SELECT 
        patient_id,
        YEAR(CURRENT_DATE) - MAX(patient_yob) AS patient_age
    GROUP BY patient_id
),

tx_enriched_final AS (
    SELECT
        t.*,

        /* ✅ bring ALL required dimensions */
        pm.territory_id,
        pm.territory AS territory_name,

        pm.payer_name,
        pm.hco_veeva_crm_id,

        pm.insurance_group,

        /* ✅ age */
        pa.patient_age

    FROM tx_enriched t

    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level pm
        ON t.patient_id = pm.patient_id

    LEFT JOIN patient_age_map pa
        ON t.patient_id = pa.patient_id
),

payer_tx_metrics AS (

SELECT
    territory_id,
    payer_name,
    hco_veeva_crm_id,

    /* ================= BASE ================= */
    COUNT(DISTINCT CASE WHEN patient_age < 17 THEN patient_id END) AS total_patients_lt17,

    COUNT(DISTINCT CASE WHEN patient_age < 17 THEN claim_id END) AS total_claims_lt17,

    /* ================= DRUG ================= */
    COUNT(DISTINCT CASE 
        WHEN patient_age < 17 AND drug_type='ELAPRASE' 
        THEN patient_id END) AS elaprase_pt_ct_lt17,

    COUNT(DISTINCT CASE 
        WHEN patient_age < 17 AND drug_type='AVLAYAH' 
        THEN patient_id END) AS avlayah_pt_ct_lt17,

    COUNT(DISTINCT CASE 
        WHEN drug_type='ELAPRASE' 
        THEN claim_id END) AS elaprase_claims_ct,

    COUNT(DISTINCT CASE 
        WHEN drug_type='AVLAYAH' 
        THEN claim_id END) AS avlayah_claims_ct,

    /* ================= DENIAL ================= */
    ROUND(
        100.0 * SUM(CASE WHEN drug_type='ELAPRASE' AND is_denial=1 THEN 1 ELSE 0 END)
        / NULLIF(COUNT(CASE WHEN drug_type='ELAPRASE' THEN claim_id END),0)
    ,2) AS elaprase_denial_rate,

    ROUND(
        100.0 * SUM(CASE WHEN drug_type='AVLAYAH' AND is_denial=1 THEN 1 ELSE 0 END)
        / NULLIF(COUNT(CASE WHEN drug_type='AVLAYAH' THEN claim_id END),0)
    ,2) AS avlayah_denial_rate

FROM tx_enriched_final   -- ✅ FIXED

GROUP BY
    territory_id,
    payer_name,
    hco_veeva_crm_id
)

SELECT 
    p.*,

    t.total_patients_lt17,
    t.total_claims_lt17,

    t.elaprase_pt_ct_lt17,
    t.avlayah_pt_ct_lt17,

    t.elaprase_claims_ct,
    t.avlayah_claims_ct,

    t.elaprase_denial_rate,
    t.avlayah_denial_rate

FROM cmpa_insights_internal_schema.PAYER_360_HCO_DETAIL p

LEFT JOIN payer_tx_metrics t
ON p.territory_id = t.territory_id
AND p.payer_name = t.payer_name
AND p.hco_veeva_crm_id = t.hco_veeva_crm_id;

In [0]:
select * from cmpa_insights_internal_schema.PAYER_360_HCO_DETAIL ;

In [0]:
select * from cmpa_insights_internal_schema.patient360_master_enriched;

# Archive

In [0]:
-- CREATE OR REPLACE TEMP VIEW patient_current_age AS
-- SELECT distinct
--     p.patient_id,
--     YEAR(CURRENT_DATE) - YEAR(d.PATIENT_YOB) AS CURRENT_AGE
-- FROM com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level p
-- LEFT JOIN com_edp_prd.com_raw.kom_patient_demographics d
--     ON p.patient_id = d.PATIENT_ID
-- WHERE d.PATIENT_YOB IS NOT NULL;


In [0]:
-- CREATE OR REPLACE TEMP VIEW patient_claim_metrics_with_rates AS
-- SELECT distinct
--     patient_id,

--     -- All claims (DX + RX)
--     COUNT(DISTINCT claim_id) AS TOTAL_CLAIMS,

--     -- Pharmacy-only denominator
--     COUNT(DISTINCT CASE
--         WHEN CLAIM_SOURCE = 'PHARMACY'
--         THEN claim_id
--     END) AS PHARMACY_TOTAL_CLAIMS,

--     COUNT(DISTINCT CASE
--         WHEN CLAIM_SOURCE = 'PHARMACY'
--          AND UPPER(TRANSACTION_STATUS) = 'PAID'
--         THEN claim_id
--     END) AS APPROVED_FILLS,

--     COUNT(DISTINCT CASE
--         WHEN CLAIM_SOURCE = 'PHARMACY'
--          AND UPPER(TRANSACTION_STATUS) = 'REJECTED'
--         THEN claim_id
--     END) AS REJECTED_FILLS,

--     COUNT(DISTINCT CASE
--         WHEN CLAIM_SOURCE = 'PHARMACY'
--          AND UPPER(TRANSACTION_STATUS) = 'REVERSED'
--         THEN claim_id
--     END) AS REVERSED_FILLS,

--     -- ✅ Correct rates (Pharmacy denominator)
--     ROUND(
--         CASE
--             WHEN COUNT(DISTINCT CASE WHEN CLAIM_SOURCE = 'PHARMACY' THEN claim_id END) = 0
--             THEN 0
--             ELSE 100.0 *
--                  COUNT(DISTINCT CASE
--                      WHEN CLAIM_SOURCE = 'PHARMACY'
--                       AND UPPER(TRANSACTION_STATUS) = 'PAID'
--                      THEN claim_id
--                  END)
--                  /
--                  COUNT(DISTINCT CASE
--                      WHEN CLAIM_SOURCE = 'PHARMACY'
--                      THEN claim_id
--                  END)
--         END
--     , 2) AS ELAPRASE_APPROVAL_RATE,

--     ROUND(
--         CASE
--             WHEN COUNT(DISTINCT CASE WHEN CLAIM_SOURCE = 'PHARMACY' THEN claim_id END) = 0
--             THEN 0
--             ELSE 100.0 *
--                  COUNT(DISTINCT CASE
--                      WHEN CLAIM_SOURCE = 'PHARMACY'
--                       AND UPPER(TRANSACTION_STATUS) = 'REJECTED'
--                      THEN claim_id
--                  END)
--                  /
--                  COUNT(DISTINCT CASE
--                      WHEN CLAIM_SOURCE = 'PHARMACY'
--                      THEN claim_id
--                  END)
--         END
--     , 2) AS ELAPRASE_REJECTION_RATE,

--     ROUND(
--         CASE
--             WHEN COUNT(DISTINCT CASE WHEN CLAIM_SOURCE = 'PHARMACY' THEN claim_id END) = 0
--             THEN 0
--             ELSE 100.0 *
--                  COUNT(DISTINCT CASE
--                      WHEN CLAIM_SOURCE = 'PHARMACY'
--                       AND UPPER(TRANSACTION_STATUS) = 'REVERSED'
--                      THEN claim_id
--                  END)
--                  /
--                  COUNT(DISTINCT CASE
--                      WHEN CLAIM_SOURCE = 'PHARMACY'
--                      THEN claim_id
--                  END)
--         END
--     , 2) AS ELAPRASE_REVERSED_RATE

-- FROM all_patient_claims
-- GROUP BY patient_id;


In [0]:
-- CREATE OR REPLACE TEMP VIEW payer_master_patient_level_extended AS
-- SELECT distinct
--     p.*,

--     -- New Patient Flags
--     COALESCE(n.NEW_PATIENT_R1M, 0)  AS NEW_PATIENT_R1M,
--     COALESCE(n.NEW_PATIENT_R3M, 0)  AS NEW_PATIENT_R3M,
--     n.FIRST_EVENT_DATE,

--     -- Claim Metrics
--     COALESCE(c.TOTAL_CLAIMS, 0)           AS TOTAL_CLAIMS,
--     COALESCE(c.PHARMACY_TOTAL_CLAIMS, 0)  AS PHARMACY_TOTAL_CLAIMS,
--     COALESCE(c.APPROVED_FILLS, 0)         AS APPROVED_FILLS,
--     COALESCE(c.REJECTED_FILLS, 0)         AS REJECTED_FILLS,
--     COALESCE(c.REVERSED_FILLS, 0)         AS REVERSED_FILLS,
--     COALESCE(c.ELAPRASE_APPROVAL_RATE, 0) AS ELAPRASE_APPROVAL_RATE,
--     COALESCE(c.ELAPRASE_REJECTION_RATE, 0) AS ELAPRASE_REJECTION_RATE,
--     COALESCE(c.ELAPRASE_REVERSED_RATE, 0)  AS ELAPRASE_REVERSED_RATE,

--     -- Age
--     a.CURRENT_AGE,
--     CASE WHEN a.CURRENT_AGE < 5 THEN 1 ELSE 0 END AS AGE_LT_5_YRS,
--     CASE WHEN a.CURRENT_AGE BETWEEN 5 AND 10 THEN 1 ELSE 0 END AS AGE_5_TO_10_YRS,
--     CASE WHEN a.CURRENT_AGE BETWEEN 11 AND 18 THEN 1 ELSE 0 END AS AGE_11_TO_18_YRS,
--     CASE WHEN a.CURRENT_AGE > 18 THEN 1 ELSE 0 END AS AGE_GT_18_YRS,

--     -- Engagement (Direct Join – No Mapping Table)
--     COALESCE(i.PIE_COMPLETED, 'NO') AS PIE_COMPLETED,
--     COALESCE(i.ACCOUNT_DIRECTOR, '-') AS ACCOUNT_DIRECTOR

-- FROM com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level p

-- LEFT JOIN new_patient_flags n
--     ON p.patient_id = n.patient_id

-- LEFT JOIN patient_claim_metrics_with_rates c
--     ON p.patient_id = c.patient_id

-- LEFT JOIN patient_current_age a
--     ON p.patient_id = a.patient_id

-- LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.payer_info i
--     ON UPPER(p.PAYER_NAME) = UPPER(i.PAYER_ACCOUNT_NAME);

-- ---------- Validation
-- SELECT distinct * FROM payer_master_patient_level_extended;


In [0]:
-- CREATE OR REPLACE TEMP VIEW payer_territory_patient_summary AS
-- SELECT DISTINCT
--     territory_id,
--     territory AS territory_name,
--     region_id,
--     region AS region_name,

--     PAYER_ID,
--     PAYER_NAME,
--     PARENT_ID,
--     PARENT_NAME,

--     -- Patient Counts
--     COUNT(DISTINCT patient_id) AS TOTAL_PATIENTS,

--     -- Insurance Split
--     COUNT(DISTINCT CASE WHEN INSURANCE_GROUP = 'MEDICARE' THEN patient_id END)   AS MEDICARE_PATIENTS,
--     COUNT(DISTINCT CASE WHEN INSURANCE_GROUP = 'MEDICAID' THEN patient_id END)   AS MEDICAID_PATIENTS,
--     COUNT(DISTINCT CASE WHEN INSURANCE_GROUP = 'COMMERCIAL' THEN patient_id END) AS COMMERCIAL_PATIENTS,
--     COUNT(DISTINCT CASE
--         WHEN INSURANCE_GROUP IS NULL
--           OR INSURANCE_GROUP NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
--         THEN patient_id END) AS OTHER_PATIENTS,

--     -- New Patients
--     SUM(NEW_PATIENT_R1M) AS NEW_PATIENTS_R1M,
--     SUM(NEW_PATIENT_R3M) AS NEW_PATIENTS_R3M,

--     -- Age Buckets
--     SUM(AGE_LT_5_YRS)     AS AGE_LT_5_YRS,
--     SUM(AGE_5_TO_10_YRS)  AS AGE_5_TO_10_YRS,
--     SUM(AGE_11_TO_18_YRS) AS AGE_11_TO_18_YRS,
--     SUM(AGE_GT_18_YRS)    AS AGE_GT_18_YRS,

--     -- Provider Counts
--     COUNT(DISTINCT hcp_npi) AS TOTAL_primary_HCPS,
--     COUNT(DISTINCT hco_npi) AS TOTAL_primary_HCOS

-- FROM payer_master_patient_level_extended
-- GROUP BY
--     territory_id,
--     territory,
--     region_id,
--     region,
--     PAYER_ID,
--     PAYER_NAME,
--     PARENT_ID,
--     PARENT_NAME;


--   ---------- Validation
-- SELECT distinct * FROM payer_territory_patient_summary;



In [0]:
-- CREATE OR REPLACE TEMP VIEW payer_master_patient_level_test AS

-- WITH eligible_patient_universe AS (
--   SELECT DISTINCT patient_id 
--   FROM eligible_patients
-- ),

-- patient_claim_counts AS (
--   SELECT DISTINCT patient_id, COUNT(DISTINCT claim_id) AS claims_count
--   FROM all_patient_claims
--   GROUP BY 1
-- ),

-- eligible_patients_with_claims AS (
--   SELECT DISTINCT a.patient_id, b.claims_count
--   FROM eligible_patient_universe a 
--   LEFT JOIN patient_claim_counts b 
--     ON a.patient_id = b.patient_id
-- ),

-- patients_with_primary_hcp AS (
--   SELECT DISTINCT a.*, b.* EXCEPT(b.patient_id)
--   FROM eligible_patients_with_claims a
--   LEFT JOIN primary_hcp b 
--     ON a.patient_id = b.patient_id
-- ),

-- patients_with_territory_region AS (
--   SELECT DISTINCT
--     a.*, 
--     b.territory_id, 
--     b.territory_name AS territory, 
--     b.region_id, 
--     b.region_name AS region
--   FROM patients_with_primary_hcp a
--   LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping b
--     ON a.hcp_zip = b.zipcode
-- ),

-- latest_plan_per_patient AS (
--   SELECT patient_id, plan_id
--   FROM (
--     SELECT DISTINCT
--       patient_id, 
--       plan_id,
--       ROW_NUMBER() OVER (
--         PARTITION BY patient_id 
--         ORDER BY fill_date DESC, npi ASC
--       ) AS rn
--     FROM all_patient_claims
--     WHERE plan_id IS NOT NULL
--   )
--   WHERE rn = 1
-- ),

-- patients_with_latest_plan AS (
--   SELECT a.*, b.plan_id
--   FROM patients_with_territory_region a
--   LEFT JOIN latest_plan_per_patient b
--     ON a.patient_id = b.patient_id
-- ),

-- patients_with_payer_attributes AS (
--   SELECT DISTINCT  
--     a.*, 
--     b.PAYER_ID, 
--     b.PAYER_NAME, 
--     b.PARENT_ID, 
--     b.PARENT_NAME, 
--     b.INSURANCE_SEGMENT, 
--     b.INSURANCE_GROUP
--   FROM patients_with_latest_plan a
--   LEFT JOIN com_edp_prd.com_raw.kom_plans b
--     ON a.plan_id = b.KH_PLAN_ID
-- ),

-- -- =========================
-- -- NEW PATIENT FLAGS
-- -- =========================
-- new_patient_flags AS (
--   SELECT DISTINCT
--     patient_id,
--     MIN(fill_date) AS FIRST_EVENT_DATE,
--     CASE WHEN MIN(fill_date) >= DATEADD(month, -1, DATE('${end_date}')) THEN 1 ELSE 0 END AS NEW_PATIENT_R1M,
--     CASE WHEN MIN(fill_date) >= DATEADD(month, -3, DATE('${end_date}')) THEN 1 ELSE 0 END AS NEW_PATIENT_R3M
--   FROM all_patient_claims
--   GROUP BY patient_id
-- ),

-- -- =========================
-- -- CLAIM METRICS
-- -- =========================
-- patient_claim_metrics AS (
--   SELECT DISTINCT
--     patient_id,
--     COUNT(DISTINCT claim_id) AS TOTAL_CLAIMS,
--     COUNT(DISTINCT CASE WHEN CLAIM_SOURCE='PHARMACY' THEN claim_id END) AS PHARMACY_TOTAL_CLAIMS,
--     COUNT(DISTINCT CASE WHEN CLAIM_SOURCE='PHARMACY' AND UPPER(TRANSACTION_STATUS)='PAID' THEN claim_id END) AS APPROVED_FILLS,
--     COUNT(DISTINCT CASE WHEN CLAIM_SOURCE='PHARMACY' AND UPPER(TRANSACTION_STATUS)='REJECTED' THEN claim_id END) AS REJECTED_FILLS,
--     COUNT(DISTINCT CASE WHEN CLAIM_SOURCE='PHARMACY' AND UPPER(TRANSACTION_STATUS)='REVERSED' THEN claim_id END) AS REVERSED_FILLS
--   FROM all_patient_claims
--   GROUP BY patient_id
-- ),

-- -- =========================
-- -- AGE
-- -- =========================
-- patient_age AS (
--   SELECT
--     p.patient_id,
--     YEAR(CURRENT_DATE) - YEAR(d.PATIENT_YOB) AS CURRENT_AGE
--   FROM patients_with_payer_attributes p
--   LEFT JOIN com_edp_prd.com_raw.kom_patient_demographics d
--     ON p.patient_id = d.PATIENT_ID
-- )

-- -- =========================
-- -- FINAL SELECT
-- -- =========================
-- SELECT DISTINCT
--   p.*,

--   -- New Patient
--   COALESCE(n.NEW_PATIENT_R1M,0) AS NEW_PATIENT_R1M,
--   COALESCE(n.NEW_PATIENT_R3M,0) AS NEW_PATIENT_R3M,
--   n.FIRST_EVENT_DATE,

--   -- Claims
--   COALESCE(c.TOTAL_CLAIMS,0) AS TOTAL_CLAIMS,
--   COALESCE(c.PHARMACY_TOTAL_CLAIMS,0) AS PHARMACY_TOTAL_CLAIMS,
--   COALESCE(c.APPROVED_FILLS,0) AS APPROVED_FILLS,
--   COALESCE(c.REJECTED_FILLS,0) AS REJECTED_FILLS,
--   COALESCE(c.REVERSED_FILLS,0) AS REVERSED_FILLS,

--   -- Rates
--   CASE WHEN c.PHARMACY_TOTAL_CLAIMS=0 THEN 0
--        ELSE ROUND(100.0*c.APPROVED_FILLS/c.PHARMACY_TOTAL_CLAIMS,2)
--   END AS ELAPRASE_APPROVAL_RATE,

--   CASE WHEN c.PHARMACY_TOTAL_CLAIMS=0 THEN 0
--        ELSE ROUND(100.0*c.REJECTED_FILLS/c.PHARMACY_TOTAL_CLAIMS,2)
--   END AS ELAPRASE_REJECTION_RATE,

--   CASE WHEN c.PHARMACY_TOTAL_CLAIMS=0 THEN 0
--        ELSE ROUND(100.0*c.REVERSED_FILLS/c.PHARMACY_TOTAL_CLAIMS,2)
--   END AS ELAPRASE_REVERSED_RATE,

--   -- Age
--   a.CURRENT_AGE,
--   CASE WHEN a.CURRENT_AGE < 5 THEN 1 ELSE 0 END AS AGE_LT_5_YRS,
--   CASE WHEN a.CURRENT_AGE BETWEEN 5 AND 10 THEN 1 ELSE 0 END AS AGE_5_TO_10_YRS,
--   CASE WHEN a.CURRENT_AGE BETWEEN 11 AND 18 THEN 1 ELSE 0 END AS AGE_11_TO_18_YRS,
--   CASE WHEN a.CURRENT_AGE > 18 THEN 1 ELSE 0 END AS AGE_GT_18_YRS,

--   -- Engagement
--   COALESCE(i.PIE_COMPLETED,'NO') AS PIE_COMPLETED,
--   COALESCE(i.ACCOUNT_DIRECTOR,'-') AS ACCOUNT_DIRECTOR

-- FROM patients_with_payer_attributes p
-- LEFT JOIN new_patient_flags n ON p.patient_id=n.patient_id
-- LEFT JOIN patient_claim_metrics c ON p.patient_id=c.patient_id
-- LEFT JOIN patient_age a ON p.patient_id=a.patient_id
-- LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.payer_info i
--   ON UPPER(p.PAYER_NAME)=UPPER(i.PAYER_ACCOUNT_NAME);

--   ------ VALIDATION
--   SELECT DISTINCT * FROM payer_master_patient_level_test;


In [0]:
-- -- CREATE OR REPLACE TEMP VIEW national_summary AS
-- SELECT DISTINCT

--     '-1' AS territory_id,
--     'NATIONAL_ROLLUP' AS territory_name,

--     '-1' AS region_id,
--     'NATIONAL_ROLLUP' AS region_name,

--     'NATIONAL_ROLLUP' AS PAYER_ID,
--     'NATIONAL_ROLLUP' AS PAYER_NAME,
--     'NATIONAL_ROLLUP' AS PARENT_ID,
--     'NATIONAL_ROLLUP' AS PARENT_NAME,

--     -- =====================
--     -- PATIENT COUNTS
--     -- =====================
--     COUNT(DISTINCT patient_id) AS TOTAL_PATIENTS,

--     COUNT(DISTINCT CASE WHEN INSURANCE_GROUP = 'MEDICARE' THEN patient_id END)   AS MEDICARE_PATIENTS,
--     COUNT(DISTINCT CASE WHEN INSURANCE_GROUP = 'MEDICAID' THEN patient_id END)   AS MEDICAID_PATIENTS,
--     COUNT(DISTINCT CASE WHEN INSURANCE_GROUP = 'COMMERCIAL' THEN patient_id END) AS COMMERCIAL_PATIENTS,
--     COUNT(DISTINCT CASE 
--         WHEN INSURANCE_GROUP IS NULL 
--           OR INSURANCE_GROUP NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
--         THEN patient_id 
--     END) AS OTHER_PATIENTS,

--     -- =====================
--     -- NEW PATIENTS
--     -- =====================
--     SUM(NEW_PATIENT_R1M) AS NEW_PATIENTS_R1M,
--     SUM(NEW_PATIENT_R3M) AS NEW_PATIENTS_R3M,

--     -- =====================
--     -- AGE BUCKETS
--     -- =====================
--     SUM(AGE_LT_5_YRS)     AS AGE_LT_5_YRS,
--     SUM(AGE_5_TO_10_YRS)  AS AGE_5_TO_10_YRS,
--     SUM(AGE_11_TO_18_YRS) AS AGE_11_TO_18_YRS,
--     SUM(AGE_GT_18_YRS)    AS AGE_GT_18_YRS,

--     -- =====================
--     -- CLAIM METRICS
--     -- =====================
--     SUM(TOTAL_CLAIMS) AS TOTAL_CLAIMS,
--     SUM(PHARMACY_TOTAL_CLAIMS) AS PHARMACY_TOTAL_CLAIMS,
--     SUM(APPROVED_FILLS) AS APPROVED_FILLS,
--     SUM(REJECTED_FILLS) AS REJECTED_FILLS,
--     SUM(REVERSED_FILLS) AS REVERSED_FILLS,

--     -- =====================
--     -- RATES (Correct Denominator)
--     -- =====================
--     CASE 
--         WHEN SUM(PHARMACY_TOTAL_CLAIMS) = 0 THEN 0
--         ELSE ROUND(100.0 * SUM(APPROVED_FILLS) / SUM(PHARMACY_TOTAL_CLAIMS), 2)
--     END AS ELAPRASE_APPROVAL_RATE,

--     CASE 
--         WHEN SUM(PHARMACY_TOTAL_CLAIMS) = 0 THEN 0
--         ELSE ROUND(100.0 * SUM(REJECTED_FILLS) / SUM(PHARMACY_TOTAL_CLAIMS), 2)
--     END AS ELAPRASE_REJECTION_RATE,

--     CASE 
--         WHEN SUM(PHARMACY_TOTAL_CLAIMS) = 0 THEN 0
--         ELSE ROUND(100.0 * SUM(REVERSED_FILLS) / SUM(PHARMACY_TOTAL_CLAIMS), 2)
--     END AS ELAPRASE_REVERSED_RATE,

--     -- =====================
--     -- PROVIDER COUNTS
--     -- =====================
--     COUNT(DISTINCT hcp_npi) AS TOTAL_primary_HCPS,
--     COUNT(DISTINCT hco_npi) AS TOTAL_primary_HCOS,

--     "-" AS PIE_COMPLETED,
--     "-" AS ACCOUNT_DIRECTOR

-- FROM payer_master_patient_level_TEST;


In [0]:
-- -- CREATE OR REPLACE TEMP VIEW payer360_master AS

-- -- ============================================================
-- -- 1️⃣ NEW PATIENT FLAGS
-- -- ============================================================
-- WITH new_patient_flags AS (
--   SELECT
--     patient_id,
--     MIN(fill_date) AS FIRST_EVENT_DATE,
--     CASE WHEN MIN(fill_date) >= DATEADD(month,-1,DATE('${end_date}')) THEN 1 ELSE 0 END AS NEW_PATIENT_R1M,
--     CASE WHEN MIN(fill_date) >= DATEADD(month,-3,DATE('${end_date}')) THEN 1 ELSE 0 END AS NEW_PATIENT_R3M
--   FROM all_patient_claims
--   GROUP BY patient_id
-- ),

-- -- ============================================================
-- -- 2️⃣ CLAIM METRICS (🔥 FIXED PHARMACY TOTAL)
-- -- ============================================================
-- patient_claim_metrics AS (
--   SELECT
--     p.patient_id,

--     -- Total claims unchanged
--     COUNT(DISTINCT a.claim_id) AS TOTAL_CLAIMS,

--     -- Pharmacy total taken directly from raw pharmacy table
--     COUNT(DISTINCT ph.PHARMACY_EVENT_ID) AS PHARMACY_TOTAL_CLAIMS,

--     COUNT(DISTINCT CASE WHEN UPPER(ph.TRANSACTION_RESULT)='PAID'
--                     THEN ph.PHARMACY_EVENT_ID END) AS APPROVED_FILLS,

--     COUNT(DISTINCT CASE WHEN UPPER(ph.TRANSACTION_RESULT)='REJECTED'
--                         THEN ph.PHARMACY_EVENT_ID END) AS REJECTED_FILLS,

--     COUNT(DISTINCT CASE WHEN UPPER(ph.TRANSACTION_RESULT)='REVERSED'
--                         THEN ph.PHARMACY_EVENT_ID END) AS REVERSED_FILLS

--   FROM com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level p

--   LEFT JOIN all_patient_claims a
--     ON p.patient_id = a.patient_id

--   LEFT JOIN com_edp_prd.com_raw.kom_pharmacy_events ph
--     ON p.patient_id = ph.PATIENT_ID
--    AND ph.FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
--    AND (
--         ph.DIAGNOSIS_CODE IN ('E761','E763')
--         OR ph.NDC11 IN ('54092070001','540920700')
--        )

--   GROUP BY p.patient_id
-- ),

-- -- ============================================================
-- -- 3️⃣ AGE
-- -- ============================================================
-- patient_age AS (
--   SELECT
--     p.patient_id,
--     YEAR(CURRENT_DATE) - YEAR(d.PATIENT_YOB) AS CURRENT_AGE
--   FROM com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level p
--   LEFT JOIN com_edp_prd.com_raw.kom_patient_demographics d
--     ON p.patient_id = d.PATIENT_ID
-- ),

-- -- ============================================================
-- -- 4️⃣ BASE ENRICHED (UNCHANGED)
-- -- ============================================================
-- base_enriched AS (
--   SELECT
--     p.*,

--     COALESCE(i.PIE_COMPLETED,'NO') AS PIE_COMPLETED,
--     COALESCE(i.ACCOUNT_DIRECTOR,'-') AS ACCOUNT_DIRECTOR,

--     COALESCE(n.NEW_PATIENT_R1M,0) AS NEW_PATIENT_R1M,
--     COALESCE(n.NEW_PATIENT_R3M,0) AS NEW_PATIENT_R3M,

--     COALESCE(c.TOTAL_CLAIMS,0)          AS TOTAL_CLAIMS,
--     COALESCE(c.PHARMACY_TOTAL_CLAIMS,0) AS PHARMACY_TOTAL_CLAIMS,
--     COALESCE(c.APPROVED_FILLS,0)        AS APPROVED_FILLS,
--     COALESCE(c.REJECTED_FILLS,0)        AS REJECTED_FILLS,
--     COALESCE(c.REVERSED_FILLS,0)        AS REVERSED_FILLS,

--     a.CURRENT_AGE,
--     CASE WHEN a.CURRENT_AGE < 5 THEN 1 ELSE 0 END AS AGE_LT_5_YRS,
--     CASE WHEN a.CURRENT_AGE BETWEEN 5 AND 10 THEN 1 ELSE 0 END AS AGE_5_TO_10_YRS,
--     CASE WHEN a.CURRENT_AGE BETWEEN 11 AND 18 THEN 1 ELSE 0 END AS AGE_11_TO_18_YRS,
--     CASE WHEN a.CURRENT_AGE > 18 THEN 1 ELSE 0 END AS AGE_GT_18_YRS

--   FROM com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level p
--   LEFT JOIN new_patient_flags n  ON p.patient_id = n.patient_id
--   LEFT JOIN patient_claim_metrics c ON p.patient_id = c.patient_id
--   LEFT JOIN patient_age a ON p.patient_id = a.patient_id
--   LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.payer_info i
--     ON UPPER(p.PAYER_NAME) = UPPER(i.PAYER_ACCOUNT_NAME)
-- ),

-- -- ============================================================
-- -- 5️⃣ ROLLUP 
-- -- ============================================================
-- rollup_metrics AS (

--   SELECT
--     COALESCE(CAST(b.territory_id AS STRING), 'ALL Territories') AS territory_id,
--     COALESCE(b.territory, 'All Territories')                    AS territory_name,
--     COALESCE(CAST(b.payer_id AS STRING), 'ALL Payers')          AS payer_id,
--     COALESCE(b.payer_name, 'All Payers')                        AS payer_name,
--     COALESCE(CAST(b.parent_id AS STRING), 'ALL Parents')        AS parent_id,
--     COALESCE(b.parent_name, 'All Parents')                      AS parent_name,

--     COUNT(DISTINCT b.patient_id) AS total_elaprase_patients,

--     COUNT(DISTINCT CASE WHEN b.insurance_group='MEDICARE' THEN b.patient_id END) AS medicare_patients,
--     COUNT(DISTINCT CASE WHEN b.insurance_group='MEDICAID' THEN b.patient_id END) AS medicaid_patients,
--     COUNT(DISTINCT CASE WHEN b.insurance_group='COMMERCIAL' THEN b.patient_id END) AS commercial_patients,
--     COUNT(DISTINCT CASE WHEN b.insurance_group IS NULL 
--         OR b.insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
--         THEN b.patient_id END) AS other_patients,

--     SUM(b.NEW_PATIENT_R1M) AS NEW_ELAPRASE_PATIENTS_R1M,
--     SUM(b.NEW_PATIENT_R3M) AS NEW_ELAPRASE_PATIENTS_R3M,

--     SUM(b.AGE_LT_5_YRS)     AS AGE_LT_5_YRS,
--     SUM(b.AGE_5_TO_10_YRS)  AS AGE_5_TO_10_YRS,
--     SUM(b.AGE_11_TO_18_YRS) AS AGE_11_TO_18_YRS,
--     SUM(b.AGE_GT_18_YRS)    AS AGE_GT_18_YRS,

--     -- ✅ Primary provider counts (patient-level)
--     COUNT(DISTINCT b.hcp_npi) AS TOTAL_PRIMARY_HCPS,
--     COUNT(DISTINCT b.hco_npi) AS TOTAL_PRIMARY_HCOS,

--     -- ✅ Universe provider counts (claim-level)
--     COUNT(DISTINCT u.hcp_npi) AS TOTAL_HCPS,
--     COUNT(DISTINCT u.hco_npi) AS TOTAL_HCOS,

--     SUM(b.TOTAL_CLAIMS)          AS TOTAL_CLAIMS,
--     SUM(b.PHARMACY_TOTAL_CLAIMS) AS PHARMACY_TOTAL_CLAIMS,
--     SUM(b.APPROVED_FILLS)        AS APPROVED_FILLS,
--     SUM(b.REJECTED_FILLS)        AS REJECTED_FILLS,
--     SUM(b.REVERSED_FILLS)        AS REVERSED_FILLS,

--     CASE WHEN SUM(b.PHARMACY_TOTAL_CLAIMS)=0 THEN 0
--          ELSE ROUND(100.0*SUM(b.APPROVED_FILLS)/SUM(b.PHARMACY_TOTAL_CLAIMS),2)
--     END AS ELAPRASE_APPROVAL_RATE,

--     CASE WHEN SUM(b.PHARMACY_TOTAL_CLAIMS)=0 THEN 0
--          ELSE ROUND(100.0*SUM(b.REJECTED_FILLS)/SUM(b.PHARMACY_TOTAL_CLAIMS),2)
--     END AS ELAPRASE_REJECTION_RATE,

--     CASE WHEN SUM(b.PHARMACY_TOTAL_CLAIMS)=0 THEN 0
--          ELSE ROUND(100.0*SUM(b.REVERSED_FILLS)/SUM(b.PHARMACY_TOTAL_CLAIMS),2)
--     END AS ELAPRASE_REVERSED_RATE,

--     CASE
--       WHEN GROUPING(b.territory_id)=0 AND GROUPING(b.payer_id)=0 THEN MAX(b.PIE_COMPLETED)
--       WHEN GROUPING(b.territory_id)=1 AND GROUPING(b.payer_id)=0 THEN MAX(b.PIE_COMPLETED)
--       ELSE 'ALL Payers'
--     END AS PIE_COMPLETED,

--     CASE
--       WHEN GROUPING(b.territory_id)=0 AND GROUPING(b.payer_id)=0 THEN MAX(b.ACCOUNT_DIRECTOR)
--       WHEN GROUPING(b.territory_id)=1 AND GROUPING(b.payer_id)=0 THEN MAX(b.ACCOUNT_DIRECTOR)
--       ELSE 'ALL Payers'
--     END AS ACCOUNT_DIRECTOR,

--     CASE
--       WHEN GROUPING(b.territory_id)=0 AND GROUPING(b.payer_id)=0 THEN 'TERRITORY_PAYER'
--       WHEN GROUPING(b.territory_id)=1 AND GROUPING(b.payer_id)=0 THEN 'PAYER_ALL_TERRITORY'
--       WHEN GROUPING(b.territory_id)=0 AND GROUPING(b.payer_id)=1 THEN 'TERRITORY_ALL_PAYER'
--       WHEN GROUPING(b.territory_id)=1 AND GROUPING(b.payer_id)=1 THEN 'NATIONAL'
--     END AS rollup_level

--   FROM base_enriched b
--   LEFT JOIN elaprase_provider_universe u
--     ON b.patient_id = u.patient_id

--   GROUP BY GROUPING SETS (
--     (b.territory_id, b.territory, b.payer_id, b.payer_name, b.parent_id, b.parent_name),
--     (b.payer_id, b.payer_name, b.parent_id, b.parent_name),
--     (b.territory_id, b.territory),
--     ()
--   )
-- )
-- ,
-- final_with_lives AS (

--   SELECT
--     r.*,
--     t.total_lives

--   FROM rollup_metrics r

--   LEFT JOIN total_lives t
--     ON r.rollup_level = t.rollup_level
--    AND r.territory_id = t.territory_id
--    AND r.payer_id     = t.payer_id
-- )

-- SELECT *
-- FROM final_with_lives
-- ORDER BY rollup_level, territory_name, payer_name;

In [0]:
-- ---------- QC8 - Chekcing patient count in all three tables

-- SELECT
--     m.total_elaprase_patients AS master_patients,
--     h.patient_count AS hcp_patients,
--     c.patient_count AS hco_patients
-- FROM cmpa_insights_internal_schema.payer360_master m
-- JOIN cmpa_insights_internal_schema.PAYER_360_HCP_DETAIL h
--   ON m.rollup_level = h.rollup_level
-- JOIN cmpa_insights_internal_schema.PAYER_360_HCO_DETAIL c
--   ON m.rollup_level = c.rollup_level
-- WHERE m.rollup_level = 'NATIONAL'
--   AND h.rollup_level = 'NATIONAL'
--   AND c.rollup_level = 'NATIONAL';